In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Helper to compute metadata for DataFrames with a name column
def compute_meta_with_name(df, name_col):
    # ensure timestamp -> timezone-aware datetime in UTC+8
    ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # attach to df for samples
    df = df.copy()
    df['_ts_dt'] = ts
    rows = len(df)
    min_ts = ts.min() if rows>0 else pd.NaT
    max_ts = ts.max() if rows>0 else pd.NaT
    unique_cmdb = pd.Series(df['cmdb_id'].dropna().unique()).astype(str)
    unique_cmdb_sample = unique_cmdb.head(20).tolist()
    unique_cmdb_count = int(unique_cmdb.shape[0])
    unique_names = pd.Series(df[name_col].dropna().unique()).astype(str)
    unique_name_sample = unique_names.head(20).tolist()
    unique_name_count = int(unique_names.shape[0])
    meta = pd.DataFrame([{
        'file': name_col.split('_')[0] + '.csv',
        'rows': rows,
        'min_ts_Asia/Shanghai': min_ts,
        'max_ts_Asia/Shanghai': max_ts,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_cmdb_sample(up_to_20)': unique_cmdb_sample,
        'unique_name_count': unique_name_count,
        'unique_name_sample(up_to_20)': unique_name_sample
    }])
    # sample rows (first 5) with converted timestamp, keep compact columns
    sample = df.head(5).copy()
    # Keep timestamp, converted ts, cmdb_id, name_col, value (if exists)
    cols = ['timestamp', '_ts_dt', 'cmdb_id', name_col]
    if 'value' in sample.columns:
        cols = cols + ['value']
    # select only existing cols
    cols = [c for c in cols if c in sample.columns]
    sample = sample[cols]
    return meta, sample

# Helper to compute metadata for error_logs (no name column)
def compute_meta_error_logs(df):
    ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    df = df.copy()
    df['_ts_dt'] = ts
    rows = len(df)
    min_ts = ts.min() if rows>0 else pd.NaT
    max_ts = ts.max() if rows>0 else pd.NaT
    unique_cmdb = pd.Series(df['cmdb_id'].dropna().unique()).astype(str)
    unique_cmdb_sample = unique_cmdb.head(20).tolist()
    unique_cmdb_count = int(unique_cmdb.shape[0])
    meta = pd.DataFrame([{
        'file': 'error_logs.csv',
        'rows': rows,
        'min_ts_Asia/Shanghai': min_ts,
        'max_ts_Asia/Shanghai': max_ts,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_cmdb_sample(up_to_20)': unique_cmdb_sample
    }])
    # no sample names requested; keep compact sample of first 5 rows (timestamp, converted, cmdb_id)
    sample = df.head(5)[['timestamp','_ts_dt','cmdb_id']].copy()
    return meta, sample

# Compute metadata and samples
metric_meta, metric_sample = compute_meta_with_name(metric_df, 'kpi_name')
trace_meta, trace_sample = compute_meta_with_name(trace_df, 'trace_name')
log_meta, log_sample = compute_meta_with_name(log_df, 'log_name')
error_meta, error_sample = compute_meta_error_logs(error_logs_df)

# Keep final displayed output compact: metadata rows and small samples
metric_meta, metric_sample, trace_meta, trace_sample, log_meta, log_sample, error_meta, error_sample

```
Out[1]:
```
```python
summary = (
    "Summary of loaded telemetry files:\n\n"
    "metric.csv: 144,252 rows. Time range 2022-03-20 16:00:00+08 to 2022-03-20 16:29:00+08. "
    "59 unique cmdb_id values (sample: adservice, adservice-0, adservice-1, adservice-2, cartservice, ...). "
    "1,152 unique KPI names (sample: app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, ...). "
    "First 5 sample rows are for 'adservice' at timestamp 2022-03-20 16:00:00+08 with values [238.0, 0.002315, 100.0, 53.361345, 24.0] across the listed KPI names.\n\n"
    "trace.csv: 36,528 rows. Time range 2022-03-20 16:00:00+08 to 2022-03-20 16:29:00+08. "
    "40 unique cmdb_id values (sample: adservice-0, adservice-1, adservice-2, ...). "
    "200 unique trace names (sample: trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, ...). "
    "First 5 sample rows are for 'adservice-0' at 2022-03-20 16:00:00+08 with values [0.000012, 0.000017, 0.0, 27.0, 0.000012].\n\n"
    "log.csv: 1,814 rows. Time range 2022-03-20 16:00:00+08 to 2022-03-20 16:29:00+08. "
    "31 unique cmdb_id values (sample: adservice-0, adservice-1, adservice-2, cartservice-0, ...). "
    "2 unique log names (log.error_count, log.row_count). "
    "First 5 sample rows show entries such as adservice-0 and adservice-1 at 2022-03-20 16:00:00+08 with log.error_count = 0.0 and log.row_count = 1644.0 / 154.0.\n\n"
    "error_logs.csv: 2,019 rows. Time range 2022-03-20 16:00:00+08 to 2022-03-20 16:29:59+08. "
    "3 unique cmdb_id values (sample: frontend-0, frontend-1, frontend-2). "
    "First 5 sample rows show error log timestamps starting at 2022-03-20 16:00:00+08 across frontend pods.\n\n"
    "All timestamps above are presented in Asia/Shanghai (UTC+8)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      file    rows      min_ts_Asia/Shanghai      max_ts_Asia/Shanghai  unique_cmdb_count                       unique_cmdb_sample(up_to_20)  unique_name_count                       unique_name_sample(up_to_20)
0  kpi.csv  144252 2022-03-20 16:00:00+08:00 2022-03-20 16:29:00+08:00                 59  [adservice, adservice-0, adservice-1, adservic...               1152  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...,     timestamp                    _ts_dt    cmdb_id        kpi_name       value
0  1647763200 2022-03-20 16:00:00+08:00  adservice  app.grpc.count  238.000000
1  1647763200 2022-03-20 16:00:00+08:00  adservice    app.grpc.mrt    0.002315
2  1647763200 2022-03-20 16:00:00+08:00  adservice     app.grpc.rr  100.000000
3  1647763200 2022-03-20 16:00:00+08:00  adservice     app.grpc.sr   53.361345
4  1647763200 2022-03-20 16:00:00+08:00  adservice  app.http.count   24.000000,         file   rows      min_ts_Asia/Shanghai      max_ts_Asia/Shanghai  unique_cmdb_count                       unique_cmdb_sample(up_to_20)  unique_name_count                       unique_name_sample(up_to_20)
0  trace.csv  36528 2022-03-20 16:00:00+08:00 2022-03-20 16:29:00+08:00                 40  [adservice-0, adservice-1, adservice-2, adserv...                200  [trace.from_frontend-0.duration_mean, trace.fr...,     timestamp                    _ts_dt      cmdb_id                           trace_name      value
0  1647763200 2022-03-20 16:00:00+08:00  adservice-0  trace.from_frontend-0.duration_mean   0.000012
1  1647763200 2022-03-20 16:00:00+08:00  adservice-0   trace.from_frontend-0.duration_p95   0.000017
2  1647763200 2022-03-20 16:00:00+08:00  adservice-0     trace.from_frontend-0.error_rate   0.000000
3  1647763200 2022-03-20 16:00:00+08:00  adservice-0      trace.from_frontend-0.row_count  27.000000
4  1647763200 2022-03-20 16:00:00+08:00  adservice-0  trace.from_frontend-1.duration_mean   0.000012,       file  rows      min_ts_Asia/Shanghai      max_ts_Asia/Shanghai  unique_cmdb_count                       unique_cmdb_sample(up_to_20)  unique_name_count      unique_name_sample(up_to_20)
0  log.csv  1814 2022-03-20 16:00:00+08:00 2022-03-20 16:29:00+08:00                 31  [adservice-0, adservice-1, adservice-2, cartse...                  2  [log.error_count, log.row_count],     timestamp                    _ts_dt      cmdb_id         log_name   value
0  1647763200 2022-03-20 16:00:00+08:00  adservice-0  log.error_count     0.0
1  1647763200 2022-03-20 16:00:00+08:00  adservice-0    log.row_count  1644.0
2  1647763200 2022-03-20 16:00:00+08:00  adservice-1  log.error_count     0.0
3  1647763200 2022-03-20 16:00:00+08:00  adservice-1    log.row_count   154.0
4  1647763200 2022-03-20 16:00:00+08:00  adservice-2  log.error_count     0.0,              file  rows      min_ts_Asia/Shanghai      max_ts_Asia/Shanghai  unique_cmdb_count          unique_cmdb_sample(up_to_20)
0  error_logs.csv  2019 2022-03-20 16:00:00+08:00 2022-03-20 16:29:59+08:00                  3  [frontend-0, frontend-1, frontend-2],     timestamp                    _ts_dt     cmdb_id
0  1647763200 2022-03-20 16:00:00+08:00  frontend-0
1  1647763201 2022-03-20 16:00:01+08:00  frontend-0
2  1647763201 2022-03-20 16:00:01+08:00  frontend-1
3  1647763202 2022-03-20 16:00:02+08:00  frontend-2
4  1647763203 2022-03-20 16:00:03+08:00  frontend-1)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse previously loaded DataFrames: metric_df, trace_df, log_df
tz = pytz.timezone('Asia/Shanghai')

# Helper to compute grouped statistics including p95 and p5
def compute_group_stats(df, group_cols, value_col='value'):
    g = df.groupby(group_cols)[value_col].agg(
        count='count',
        mean='mean',
        std='std'
    ).reset_index()
    # compute percentiles separately and merge to preserve performance
    p95 = df.groupby(group_cols)[value_col].quantile(0.95).reset_index().rename(columns={value_col:'p95'})
    p5  = df.groupby(group_cols)[value_col].quantile(0.05).reset_index().rename(columns={value_col:'p5'})
    g = g.merge(p95, on=group_cols).merge(p5, on=group_cols)
    # ensure column order
    cols = group_cols + ['count','mean','std','p95','p5']
    return g[cols]

# 1) metric.csv grouped by (cmdb_id, kpi_name)
metric_groups = compute_group_stats(metric_df, ['cmdb_id','kpi_name'], 'value')
metric_group_count = int(metric_groups.shape[0])
metric_groups_top20 = metric_groups.sort_values('p95', ascending=False).head(20)

# 2) trace.csv grouped by (cmdb_id, trace_name)
trace_groups = compute_group_stats(trace_df, ['cmdb_id','trace_name'], 'value')
trace_group_count = int(trace_groups.shape[0])
trace_groups_top20 = trace_groups.sort_values('p95', ascending=False).head(20)

# 3) log.csv grouped by (cmdb_id, log_name) - return all groups (expected to be few)
log_groups = compute_group_stats(log_df, ['cmdb_id','log_name'], 'value')
log_group_count = int(log_groups.shape[0])
log_groups_sorted = log_groups.sort_values('p95', ascending=False)  # present all groups sorted by p95

# Keep outputs compact: counts and top-20 samples (or all for logs)
metric_group_count, metric_groups_top20, trace_group_count, trace_groups_top20, log_group_count, log_groups_sorted

```
Out[2]:
```
Here’s a compact plain-English summary of the computed global thresholds and group statistics.

metric.csv
- Total unique (cmdb_id, kpi_name) groups: 4,818
- Top 20 groups by P95 (columns: cmdb_id | kpi_name | count | mean | std | p95 | p5):
  1. adservice2 | runtime.java_lang_Runtime_StartTime | 30 | 1.647523e+12 | 0.0 | 1.647523e+12 | 1.647523e+12
  2. adservice  | runtime.java_lang_Runtime_StartTime | 30 | 1.647523e+12 | 0.0 | 1.647523e+12 | 1.647523e+12
  3. adservice2 | runtime.java_lang_OperatingSystem_ProcessCpuTime | 30 | 8.027248e+11 | 1.693378e+09 | 8.052562e+11 | 8.002368e+11
  4. adservice  | runtime.java_lang_OperatingSystem_ProcessCpuTime | 30 | 7.861785e+11 | 1.738315e+09 | 7.887515e+11 | 7.835904e+11
  5. adservice2 | runtime.java_lang_Threading_CurrentThreadCpuTime | 30 | 7.523221e+10 | 1.663235e+08 | 7.547548e+10 | 7.497832e+10
  6. adservice  | runtime.java_lang_Threading_CurrentThreadCpuTime | 30 | 7.422382e+10 | 1.639118e+08 | 7.446507e+10 | 7.398363e+10
  7. adservice  | runtime.java_lang_Threading_CurrentThreadUserTime | 30 | 7.219989e+10 | 1.601623e+08 | 7.243375e+10 | 7.196812e+10
  8. adservice2 | runtime.java_lang_Threading_CurrentThreadUserTime | 30 | 7.207172e+10 | 1.602981e+08 | 7.230408e+10 | 7.182608e+10
  9. adservice  | runtime.java_lang_Threading_CurrentThreadAlloc... | 30 | 5.876743e+10 | 1.288334e+08 | 5.895848e+10 | 5.857651e+10
 10. adservice2 | runtime.java_lang_Threading_CurrentThreadAlloc... | 30 | 5.865018e+10 | 1.287045e+08 | 5.884114e+10 | 5.845897e+10
 11. node-4     | system.disk.total                          | 30 | 7.616300e+09 | 0.0         | 7.616300e+09 | 7.616300e+09
 12. node-4     | system.fs.inodes.total                     | 30 | 6.665627e+09 | 2.455173e+06 | 6.668231e+09 | 6.662248e+09
 13. node-4     | system.fs.inodes.free                      | 30 | 6.663042e+09 | 2.455167e+06 | 6.665645e+09 | 6.659664e+09
 14. node-3     | system.disk.total                          | 30 | 5.867011e+09 | 0.0         | 5.867011e+09 | 5.867011e+09
 15. node-2     | system.disk.total                          | 30 | 5.867011e+09 | 0.0         | 5.867011e+09 | 5.867011e+09
 16. node-1     | system.disk.total                          | 30 | 5.867011e+09 | 0.0         | 5.867011e+09 | 5.867011e+09
 17. node-4     | system.disk.used                           | 30 | 5.753496e+09 | 6.141273e+05 | 5.754340e+09 | 5.752845e+09
 18. node-3     | system.fs.inodes.total                     | 30 | 5.009282e+09 | 1.355084e+06 | 5.011159e+09 | 5.006901e+09
 19. node-2     | system.fs.inodes.total                     | 30 | 5.009333e+09 | 1.295726e+06 | 5.011034e+09 | 5.006968e+09
 20. node-1     | system.fs.inodes.total                     | 30 | 5.009314e+09 | 1.304127e+06 | 5.011033e+09 | 5.006966e+09

trace.csv
- Total unique (cmdb_id, trace_name) groups: 1,232
- Top 20 groups by P95 (columns: cmdb_id | trace_name | count | mean | std | p95 | p5):
  1. frontend-0  | trace.self.row_count                      | 30 | 1163.70 | 70.15  | 1281.60 | 1048.45
  2. frontend-2  | trace.self.row_count                      | 30 | 1169.60 | 76.12  | 1267.05 | 1039.65
  3. frontend-1  | trace.self.row_count                      | 30 | 1024.80 | 58.43  | 1101.70 |  935.45
  4. frontend2-0 | trace.self.row_count                      | 30 |  689.73 | 86.36  |  837.50 |  567.30
  5. productcatalogservice2-0 | trace.from_frontend2-0.row_count | 30 | 318.50 | 44.11 | 389.10 | 251.00
  6. frontend2-0 | trace.to_productcatalogservice2-0.row_count | 30 | 318.50 | 44.11 | 389.10 | 251.00
  7. frontend2-0 | trace.to_currencyservice2-0.row_count    | 30 | 203.20 | 39.28  | 263.20 | 151.15
  8. currencyservice2-0 | trace.from_frontend2-0.row_count    | 30 | 203.20 | 39.28  | 263.20 | 151.15
  9. frontend-2  | trace.to_productcatalogservice-0.row_count | 30 | 179.90 | 17.42  | 210.00 | 150.95
 10. productcatalogservice-0 | trace.from_frontend-2.row_count   | 30 | 179.90 | 17.42  | 210.00 | 150.95
 11. productcatalogservice-2 | trace.from_frontend-2.row_count   | 30 | 179.8667 | 17.441 | 209.55 | 151.50
 12. productcatalogservice-1 | trace.from_frontend-2.row_count   | 30 | 179.90 | 17.383 | 209.55 | 151.05
 13. frontend-2  | trace.to_productcatalogservice-2.row_count | 30 | 179.8667 | 17.441 | 209.55 | 151.50
 14. frontend-2  | trace.to_productcatalogservice-1.row_count | 30 | 179.90 | 17.383 | 209.55 | 151.05
 15. productcatalogservice-1 | trace.from_frontend-0.row_count   | 30 | 179.5667 | 14.750 | 208.50 | 161.90
 16. frontend-0  | trace.to_productcatalogservice-1.row_count | 30 | 179.5667 | 14.750 | 208.50 | 161.90
 17. frontend-0  | trace.to_productcatalogservice-2.row_count | 30 | 179.5333 | 14.560 | 207.95 | 161.35
 18. productcatalogservice-2 | trace.from_frontend-0.row_count   | 30 | 179.5333 | 14.5596 | 207.95 | 161.35
 19. frontend-0  | trace.to_productcatalogservice-0.row_count | 30 | 179.5667 | 14.512 | 207.50 | 160.90
 20. productcatalogservice-0 | trace.from_frontend-0.row_count   | 30 | 179.5667 | 14.512 | 207.50 | 160.90

log.csv
- Total unique (cmdb_id, log_name) groups: 62
- All groups (sorted by P95). Highlights (columns: cmdb_id | log_name | count | mean | std | p95 | p5):
  - frontend-2 | log.row_count | 30 | 1802.1333 | 145.20 | 2017.90 | 1587.15
  - frontend-0 | log.row_count | 30 | 1792.9333 |  93.84 | 1934.05 | 1658.35
  - frontend-1 | log.row_count | 30 | 1583.4333 |  94.61 | 1735.70 | 1427.00
  - adservice-0 | log.row_count | 30 | 1492.6000 | 100.83 | 1650.60 | 1359.90
  - productcatalogservice-1 | log.row_count | 30 | 1203.7333 | 57.13 | 1278.05 | 1092.50
  - ... (other log.row_count groups) ...
  - Many log.error_count groups have mean = 0.0 and p95 = 0.0 (no observed errors for those groups).

Notes
- All percentiles and statistics were computed globally using the entire file for each group (per your instruction), before any time filtering.
- I kept each file’s summary compact and reported top groups where requested. Full grouped tables are stored in the kernel (variables metric_groups, trace_groups, log_groups) if you want more rows or exact ordering printed.

The original code execution output of IPython Kernel is also provided below for reference:

(4818,          cmdb_id                                           kpi_name  count          mean           std           p95            p5
829   adservice2                runtime.java_lang_Runtime_StartTime     30  1.647523e+12  0.000000e+00  1.647523e+12  1.647523e+12
250    adservice                runtime.java_lang_Runtime_StartTime     30  1.647523e+12  0.000000e+00  1.647523e+12  1.647523e+12
823   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime     30  8.027248e+11  1.693378e+09  8.052562e+11  8.002368e+11
244    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime     30  7.861785e+11  1.738315e+09  7.887515e+11  7.835904e+11
832   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime     30  7.523221e+10  1.663235e+08  7.547548e+10  7.497832e+10
253    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime     30  7.422382e+10  1.639118e+08  7.446507e+10  7.398363e+10
255    adservice  runtime.java_lang_Threading_CurrentThreadUserTime     30  7.219989e+10  1.601623e+08  7.243375e+10  7.196812e+10
834   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime     30  7.207172e+10  1.602981e+08  7.230408e+10  7.182608e+10
252    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...     30  5.876743e+10  1.288334e+08  5.895848e+10  5.857651e+10
831   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...     30  5.865018e+10  1.287045e+08  5.884114e+10  5.845897e+10
3255      node-4                                  system.disk.total     30  7.616300e+09  0.000000e+00  7.616300e+09  7.616300e+09
3259      node-4                             system.fs.inodes.total     30  6.665627e+09  2.455173e+06  6.668231e+09  6.662248e+09
3257      node-4                              system.fs.inodes.free     30  6.663042e+09  2.455167e+06  6.665645e+09  6.659664e+09
3196      node-3                                  system.disk.total     30  5.867011e+09  0.000000e+00  5.867011e+09  5.867011e+09
3138      node-2                                  system.disk.total     30  5.867011e+09  0.000000e+00  5.867011e+09  5.867011e+09
3079      node-1                                  system.disk.total     30  5.867011e+09  0.000000e+00  5.867011e+09  5.867011e+09
3256      node-4                                   system.disk.used     30  5.753496e+09  6.141273e+05  5.754340e+09  5.752845e+09
3200      node-3                             system.fs.inodes.total     30  5.009282e+09  1.355084e+06  5.011159e+09  5.006901e+09
3142      node-2                             system.fs.inodes.total     30  5.009333e+09  1.295726e+06  5.011034e+09  5.006968e+09
3083      node-1                             system.fs.inodes.total     30  5.009314e+09  1.304127e+06  5.011033e+09  5.006966e+09, 1232,                        cmdb_id                                   trace_name  count         mean        std      p95       p5
575                 frontend-0                         trace.self.row_count     30  1163.700000  70.146718  1281.60  1048.45
759                 frontend-2                         trace.self.row_count     30  1169.600000  76.117695  1267.05  1039.65
667                 frontend-1                         trace.self.row_count     30  1024.800000  58.428500  1101.70   935.45
851                frontend2-0                         trace.self.row_count     30   689.733333  86.356891   837.50   567.30
1047  productcatalogservice2-0             trace.from_frontend2-0.row_count     30   318.500000  44.108604   389.10   251.00
871                frontend2-0  trace.to_productcatalogservice2-0.row_count     30   318.500000  44.108604   389.10   251.00
867                frontend2-0        trace.to_currencyservice2-0.row_count     30   203.200000  39.279195   263.20   151.15
527         currencyservice2-0             trace.from_frontend2-0.row_count     30   203.200000  39.279195   263.20   151.15
811                 frontend-2   trace.to_productcatalogservice-0.row_count     30   179.900000  17.424418   210.00   150.95
943    productcatalogservice-0              trace.from_frontend-2.row_count     30   179.900000  17.424418   210.00   150.95
1023   productcatalogservice-2              trace.from_frontend-2.row_count     30   179.866667  17.441001   209.55   151.50
983    productcatalogservice-1              trace.from_frontend-2.row_count     30   179.900000  17.382810   209.55   151.05
819                 frontend-2   trace.to_productcatalogservice-2.row_count     30   179.866667  17.441001   209.55   151.50
815                 frontend-2   trace.to_productcatalogservice-1.row_count     30   179.900000  17.382810   209.55   151.05
975    productcatalogservice-1              trace.from_frontend-0.row_count     30   179.566667  14.750063   208.50   161.90
631                 frontend-0   trace.to_productcatalogservice-1.row_count     30   179.566667  14.750063   208.50   161.90
635                 frontend-0   trace.to_productcatalogservice-2.row_count     30   179.533333  14.559588   207.95   161.35
1015   productcatalogservice-2              trace.from_frontend-0.row_count     30   179.533333  14.559588   207.95   161.35
627                 frontend-0   trace.to_productcatalogservice-0.row_count     30   179.566667  14.512024   207.50   160.90
935    productcatalogservice-0              trace.from_frontend-0.row_count     30   179.566667  14.512024   207.50   160.90, 62,                     cmdb_id         log_name  count         mean         std      p95       p5
35               frontend-2    log.row_count     30  1802.133333  145.197543  2017.90  1587.15
31               frontend-0    log.row_count     30  1792.933333   93.838795  1934.05  1658.35
33               frontend-1    log.row_count     30  1583.433333   94.607240  1735.70  1427.00
1               adservice-0    log.row_count     30  1492.600000  100.825763  1650.60  1359.90
45  productcatalogservice-1    log.row_count     30  1203.733333   57.125351  1278.05  1092.50
..                      ...              ...    ...          ...         ...      ...      ...
48  recommendationservice-0  log.error_count     30     0.000000    0.000000     0.00     0.00
56        shippingservice-0  log.error_count     30     0.000000    0.000000     0.00     0.00
54             redis-cart-0  log.error_count      7     0.000000    0.000000     0.00     0.00
58        shippingservice-1  log.error_count     30     0.000000    0.000000     0.00     0.00
60        shippingservice-2  log.error_count     30     0.000000    0.000000     0.00     0.00

[62 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime

# reuse variables from previous steps: metric_df, metric_groups
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df has timezone-aware datetime column in UTC+8
if '_ts_dt' not in metric_df.columns:
    metric_df['_ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window: 2022-03-20 16:00:00 to 2022-03-20 16:30:00 Asia/Shanghai (start inclusive, end exclusive)
start_dt = tz.localize(datetime(2022,3,20,16,0,0))
end_dt   = tz.localize(datetime(2022,3,20,16,30,0))

# Filter metric rows to the window
metric_window = metric_df[(metric_df['_ts_dt'] >= start_dt) & (metric_df['_ts_dt'] < end_dt)].copy()

# Merge global thresholds (p95, p5) from metric_groups into metric_window
# metric_groups has columns: cmdb_id, kpi_name, count, mean, std, p95, p5
metric_window = metric_window.merge(
    metric_groups[['cmdb_id','kpi_name','p95','p5']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Mark high/low anomalies and compute per-row deviation ratios (safely handle zero percentiles)
# High anomaly: value >= p95
# Low anomaly: value <= p5
metric_window['is_high'] = metric_window['value'] >= metric_window['p95']
metric_window['is_low']  = metric_window['value'] <= metric_window['p5']

# Avoid division by zero: only compute ratio where denom > 0 and corresponding anomaly flag is True
metric_window['dev_high'] = 0.0
mask_high = metric_window['is_high'] & (metric_window['p95'] > 0)
metric_window.loc[mask_high, 'dev_high'] = (metric_window.loc[mask_high, 'value'] - metric_window.loc[mask_high, 'p95']) / metric_window.loc[mask_high, 'p95']

metric_window['dev_low'] = 0.0
mask_low = metric_window['is_low'] & (metric_window['p5'] > 0)
metric_window.loc[mask_low, 'dev_low'] = (metric_window.loc[mask_low, 'p5'] - metric_window.loc[mask_low, 'value']) / metric_window.loc[mask_low, 'p5']

# Group by (cmdb_id, kpi_name) within the window and compute required aggregates
def earliest_ts(series):
    if series.dropna().empty:
        return pd.NaT
    return series.min()

agg = metric_window.groupby(['cmdb_id','kpi_name']).agg(
    count_in_window = ('value','count'),
    anomaly_count_high = ('is_high','sum'),
    anomaly_count_low  = ('is_low','sum'),
    earliest_high_ts = ('_ts_dt', lambda s: earliest_ts(s[metric_window.loc[s.index,'is_high']])),
    earliest_low_ts  = ('_ts_dt', lambda s: earliest_ts(s[metric_window.loc[s.index,'is_low']])),
    max_dev_ratio_high = ('dev_high','max'),
    max_dev_ratio_low  = ('dev_low','max')
).reset_index()

# Replace NaT with None for compact display (but keep dtype datetime where present)
agg['earliest_high_ts'] = agg['earliest_high_ts'].where(agg['earliest_high_ts'].notna(), None)
agg['earliest_low_ts']  = agg['earliest_low_ts'].where(agg['earliest_low_ts'].notna(), None)

# Ensure max_dev ratios are numeric and fillna with 0
agg['max_dev_ratio_high'] = agg['max_dev_ratio_high'].fillna(0).astype(float)
agg['max_dev_ratio_low']  = agg['max_dev_ratio_low'].fillna(0).astype(float)

# Store full per-group anomaly summary for later steps
metric_anomaly_summary = agg.copy()

# Compute total anomaly count for sorting and get top 20 groups
metric_anomaly_summary['total_anomalies'] = metric_anomaly_summary['anomaly_count_high'] + metric_anomaly_summary['anomaly_count_low']
metric_anomaly_summary_top20 = metric_anomaly_summary.sort_values('total_anomalies', ascending=False).head(20)

# Keep output compact: select and order requested columns
cols_out = [
    'cmdb_id','kpi_name','count_in_window',
    'anomaly_count_high','anomaly_count_low',
    'earliest_high_ts','earliest_low_ts',
    'max_dev_ratio_high','max_dev_ratio_low'
]
metric_anomaly_summary_top20 = metric_anomaly_summary_top20[cols_out]

# Display the top 20 summary (variable output as required)
metric_anomaly_summary_top20

```
Out[3]:
```
Summary of anomaly counts in the incident window (2022-03-20 16:00 — 16:30 Asia/Shanghai):

- Operation performed: for every (cmdb_id, kpi_name) group we counted high-side anomalies (value >= group's P95) and low-side anomalies (value <= group's P5) inside the window, and computed earliest timestamps and max deviation ratios for each side. Full per-group results are stored in the kernel variable metric_anomaly_summary.

- Top 20 groups by total anomalies: all top entries are metrics from cmdb_id = shippingservice2-0. Each shown group has:
  - count_in_window = 30 (all 30 minutes present)
  - anomaly_count_high = 30 and anomaly_count_low = 30 (every point in the window triggers both high- and low-side condition relative to the global thresholds)
  - earliest_high_ts = 2022-03-20 16:00:00+08 and earliest_low_ts = 2022-03-20 16:00:00+08
  - max_dev_ratio_high = 0.0 and max_dev_ratio_low = 0.0 for these groups

- Interpretation: the top groups are various container/node-related KPIs for shippingservice2-0 (fs reads/writes, IO times, memory, file descriptors, ulimits, etc.). They show every sampled point flagged as both >=P95 and <=P5 but with zero reported max deviation ratios. (The stored data indicates no positive deviation ratio from the percentile thresholds in these rows.)

If you want, I can:
- show a few more rows from metric_anomaly_summary,
- compute how many groups have any anomalies,
- or inspect one of the shippingservice2-0 KPI series to explain the zero deviation ratios in more detail.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window  anomaly_count_high  anomaly_count_low          earliest_high_ts           earliest_low_ts  max_dev_ratio_high  max_dev_ratio_low
4801  shippingservice2-0  container.node-5.container_ulimits_soft.max_op...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4761  shippingservice2-0      container.node-5.container_fs_writes./dev/vda               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4762  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4763  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4764  shippingservice2-0  container.node-5.container_fs_writes_merged./d...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4766  shippingservice2-0            container.node-5.container_memory_cache               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4767  shippingservice2-0          container.node-5.container_memory_failcnt               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4753  shippingservice2-0       container.node-5.container_fs_reads./dev/vda               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4754  shippingservice2-0      container.node-5.container_fs_reads./dev/vda1               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4755  shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4756  shippingservice2-0  container.node-5.container_fs_reads_merged./de...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4757  shippingservice2-0  container.node-5.container_fs_sector_reads./de...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4758  shippingservice2-0  container.node-5.container_fs_sector_writes./d...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4760  shippingservice2-0  container.node-5.container_fs_write_seconds./d...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4745  shippingservice2-0        container.node-5.container_file_descriptors               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4746  shippingservice2-0     container.node-5.container_fs_inodes./dev/vda1               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4747  shippingservice2-0  container.node-5.container_fs_inodes_free./dev...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4748  shippingservice2-0  container.node-5.container_fs_io_current./dev/...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4749  shippingservice2-0  container.node-5.container_fs_io_time_seconds....               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0
4750  shippingservice2-0  container.node-5.container_fs_io_time_weighted...               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00                 0.0                0.0```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist in kernel: metric_groups, metric_df
# 1) All metric_groups rows for shippingservice2-0 with selected columns, sorted by kpi_name (limit 200)
shipping_metric_groups_subset = (
    metric_groups[metric_groups['cmdb_id'] == 'shippingservice2-0']
    .loc[:, ['kpi_name', 'count', 'mean', 'std', 'p95', 'p5']]
    .sort_values('kpi_name')
    .head(200)
)

# 2) Aggregate counts: how many groups have p95 == p5 and how many p95 != p5 (use isclose for float safety)
sg = metric_groups[metric_groups['cmdb_id'] == 'shippingservice2-0'].copy()
equal_mask = np.isclose(sg['p95'].fillna(0), sg['p5'].fillna(0))
num_equal = int(equal_mask.sum())
num_not_equal = int((~equal_mask).sum())
shipping_p95_equal_counts = pd.DataFrame([{
    'cmdb_id': 'shippingservice2-0',
    'groups_total': int(sg.shape[0]),
    'p95_eq_p5_count': num_equal,
    'p95_ne_p5_count': num_not_equal
}])

# 3) Sample rows from metric.csv for shippingservice2-0 in the incident window (2022-03-20 16:00 to 16:30 Asia/Shanghai)
start_dt = tz.localize(datetime(2022,3,20,16,0,0))
end_dt   = tz.localize(datetime(2022,3,20,16,30,0))

# ensure _ts_dt exists
if '_ts_dt' not in metric_df.columns:
    metric_df['_ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

shipping_window = metric_df[
    (metric_df['cmdb_id'] == 'shippingservice2-0') &
    (metric_df['_ts_dt'] >= start_dt) &
    (metric_df['_ts_dt'] < end_dt)
].loc[:, ['timestamp','_ts_dt','kpi_name','value']]

# sort by timestamp then kpi_name and limit to 20 rows for compact output
shipping_window_samples = shipping_window.sort_values(['_ts_dt','kpi_name']).head(20).drop(columns=['_ts_dt'])

# Display the three compact results
shipping_metric_groups_subset, shipping_p95_equal_counts, shipping_window_samples

```
Out[4]:
```
```python
summary = (
    "Results for cmdb_id = shippingservice2-0 (compact):\n\n"
    "1) metric_groups: 80 (cmdb_id,kpi_name) groups were found for shippingservice2-0. "
    "A sorted subset was returned (examples): "
    "container.node-5.container_cpu_cfs_periods, container.node-5.container_cpu_usage_seconds, "
    "container.node-5.container_file_descriptors, container.node-5.container_fs_limit_MB./dev/vda1, "
    "mesh.source.shippingservice2.jaeger-collector....\n\n"
    "2) Aggregate comparison of percentiles (single-line): "
    "out of 80 groups, 50 groups have p95 == p5, and 30 groups have p95 != p5.\n\n"
    "3) Sample metric rows in the incident window (2022-03-20 16:00 to 16:30 Asia/Shanghai): "
    "up to 20 sample rows were returned (sorted by timestamp, then kpi_name). "
    "Samples start at timestamp 1647763200 (2022-03-20 16:00:00+08) and include values such as:\n"
    "- container.node-5.container_cpu_cfs_periods = 12.0\n"
    "- container.node-5.container_cpu_usage_seconds = 0.013351\n"
    "- container.node-5.container_file_descriptors = 10.0\n"
    "- container.node-5.container_fs_limit_MB./dev/vda1 = 604630.738281\n"
    "and many fs read/write related metrics with value = 0.0 in that sample.\n\n"
    "Note: The high count of groups with p95 == p5 (50/80) indicates many metrics are constant or show no spread "
    "across the full-history window used to compute global thresholds. This likely explains why, for several KPIs, "
    "points evaluated equal both threshold boundaries in the incident window."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                               kpi_name  count          mean          std           p95         p5
4738         container.node-5.container_cpu_cfs_periods     30     23.100000     7.010337     33.600000     12.000
4739  container.node-5.container_cpu_cfs_throttled_p...     30      0.000000     0.000000      0.000000      0.000
4740  container.node-5.container_cpu_cfs_throttled_s...     30      0.000000     0.000000      0.000000      0.000
4741    container.node-5.container_cpu_load_average_10s     30      0.000000     0.000000      0.000000      0.000
4742      container.node-5.container_cpu_system_seconds     30      0.005500     0.004798      0.012750      0.000
...                                                 ...    ...           ...          ...           ...        ...
4813  mesh.destination.frontend2.shippingservice2.is...     30     13.633333     3.834807     19.050000      8.225
4814  mesh.source.shippingservice2.jaeger-collector....     30  40476.666667  9324.751391  52278.750000  25112.500
4815  mesh.source.shippingservice2.jaeger-collector....     30      0.000348     0.000569      0.001575      0.000
4816  mesh.source.shippingservice2.jaeger-collector....     30     13.400000     3.029738     17.275000      8.400
4817  mesh.source.shippingservice2.jaeger-collector....     30   1329.983333   305.266319   1696.475000    791.025

[80 rows x 6 columns],               cmdb_id  groups_total  p95_eq_p5_count  p95_ne_p5_count
0  shippingservice2-0            80               50               30,        timestamp                                           kpi_name          value
4726  1647763200         container.node-5.container_cpu_cfs_periods      12.000000
4727  1647763200  container.node-5.container_cpu_cfs_throttled_p...       0.000000
4728  1647763200  container.node-5.container_cpu_cfs_throttled_s...       0.000000
4729  1647763200    container.node-5.container_cpu_load_average_10s       0.000000
4730  1647763200      container.node-5.container_cpu_system_seconds       0.000000
4731  1647763200       container.node-5.container_cpu_usage_seconds       0.013351
4732  1647763200        container.node-5.container_cpu_user_seconds       0.005000
4733  1647763200        container.node-5.container_file_descriptors      10.000000
4734  1647763200     container.node-5.container_fs_inodes./dev/vda1       0.000000
4735  1647763200  container.node-5.container_fs_inodes_free./dev...       0.000000
4736  1647763200  container.node-5.container_fs_io_current./dev/...       0.000000
4737  1647763200  container.node-5.container_fs_io_time_seconds....       0.000000
4738  1647763200  container.node-5.container_fs_io_time_weighted...       0.000000
4739  1647763200   container.node-5.container_fs_limit_MB./dev/vda1  604630.738281
4740  1647763200  container.node-5.container_fs_read_seconds./de...       0.000000
4741  1647763200       container.node-5.container_fs_reads./dev/vda       0.000000
4742  1647763200      container.node-5.container_fs_reads./dev/vda1       0.000000
4743  1647763200    container.node-5.container_fs_reads_MB./dev/vda       0.000000
4744  1647763200  container.node-5.container_fs_reads_merged./de...       0.000000
4745  1647763200  container.node-5.container_fs_sector_reads./de...       0.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_anomaly_summary, metric_groups
# 1) Counts
any_anom_mask = (metric_anomaly_summary['anomaly_count_high'] > 0) | (metric_anomaly_summary['anomaly_count_low'] > 0)
positive_dev_mask = (metric_anomaly_summary['max_dev_ratio_high'] > 0) | (metric_anomaly_summary['max_dev_ratio_low'] > 0)

total_groups_with_any_anomaly = int(any_anom_mask.sum())
groups_with_positive_dev = int((any_anom_mask & positive_dev_mask).sum())

counts_df = pd.DataFrame([{
    'groups_with_any_anomaly': total_groups_with_any_anomaly,
    'groups_with_positive_max_deviation': groups_with_positive_dev
}])

# 2) Build metric_fault_candidates: join p95/p5, compute severity and earliest_anomaly_ts
# Join p95 and p5 from metric_groups
mg_small = metric_groups[['cmdb_id','kpi_name','p95','p5']].copy()
candidates = metric_anomaly_summary.merge(mg_small, on=['cmdb_id','kpi_name'], how='left')

# Compute severity and earliest_anomaly_ts
candidates['severity'] = candidates[['max_dev_ratio_high','max_dev_ratio_low']].max(axis=1)

# Normalize earliest timestamps to pd.Timestamp or NaT
def to_ts(x):
    return pd.NaT if x is None else pd.to_datetime(x)

candidates['earliest_high_ts_ts'] = candidates['earliest_high_ts'].apply(to_ts)
candidates['earliest_low_ts_ts']  = candidates['earliest_low_ts'].apply(to_ts)

candidates['earliest_anomaly_ts'] = candidates[['earliest_high_ts_ts','earliest_low_ts_ts']].min(axis=1)

# Select and order columns requested, keep full table in metric_fault_candidates
cols_out = [
    'cmdb_id','kpi_name','count_in_window',
    'anomaly_count_high','anomaly_count_low',
    'earliest_high_ts','earliest_low_ts',
    'max_dev_ratio_high','max_dev_ratio_low',
    'p95','p5',
    'earliest_anomaly_ts','severity'
]
metric_fault_candidates = candidates.copy()[cols_out]

# 3) Top 20 by severity desc, then earliest_anomaly_ts asc
metric_fault_candidates_top20 = metric_fault_candidates.sort_values(
    by=['severity','earliest_anomaly_ts'],
    ascending=[False, True]
).head(20)

# Keep output compact: show counts and top20 rows only
counts_df, metric_fault_candidates_top20

```
Out[5]:
```
Summary (compact):

1) Counts
- Total (cmdb_id, kpi_name) groups with any anomaly in the incident window: 4,818
- Of those, groups with a positive max deviation (severity > 0): 1,770

2) Top 20 fault candidates (sorted by severity = max(max_dev_ratio_high, max_dev_ratio_low), compact rows)
Format: cmdb_id | kpi_name | count_in_window | high_cnt | low_cnt | earliest_anomaly_ts | severity | p95 | p5

1. node-1 | system.io.avg_q_sz | 30 | 13 | 17 | 2022-03-20 16:00:00+08 | 25.000000 | 0.010000 | 0.000000  
2. paymentservice-1 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:00:00+08 | 20.631559 | 0.025791 | 0.018894  
3. paymentservice-2 | container.node-5.container_network_receive_MB.... | 30 | 3 | 2 | 2022-03-20 16:00:00+08 | 20.322449 | 0.025848 | 0.019963  
4. paymentservice-0 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:13:00+08 | 20.250528 | 0.025972 | 0.018995  
5. paymentservice2-0 | container.node-6.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:06:00+08 | 18.601700 | 0.028222 | 0.017363  
6. emailservice2-0 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:00:00+08 | 18.320397 | 0.028764 | 0.019818  
7. checkoutservice-1 | mesh.source.checkoutservice.jaeger-collector.i... | 30 | 2 | 21 | 2022-03-20 16:00:00+08 | 15.265625 | 0.001600 | 0.000000  
8. shippingservice-1 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:02:00+08 | 13.047716 | 0.046568 | 0.034712  
9. shippingservice2-0 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:02:00+08 | 12.079747 | 0.042903 | 0.026187  
10. checkoutservice-1 | mesh.source.checkoutservice.emailservice.istio... | 30 | 2 | 2 | 2022-03-20 16:10:00+08 | 10.947642 | 0.005539 | 0.003911  
11. shippingservice-0 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:03:00+08 | 10.511052 | 0.049265 | 0.036137  
12. emailservice-0 | mesh.destination.checkoutservice.emailservice.... | 30 | 2 | 2 | 2022-03-20 16:06:00+08 | 10.376623 | 0.005775 | 0.003675  
13. checkoutservice2-0 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:03:00+08 | 10.063825 | 0.052091 | 0.035035  
14. checkoutservice-1 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:08:00+08 | 8.413486 | 0.062166 | 0.042268  
15. adservice-2 | container.node-5.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:20:00+08 | 7.441061 | 0.074807 | 0.057934  
16. node-3 | system.io.avg_q_sz | 30 | 2 | 28 | 2022-03-20 16:00:00+08 | 7.363636 | 0.027500 | 0.000000  
17. adservice2-0 | container.node-6.container_network_receive_MB.... | 30 | 2 | 2 | 2022-03-20 16:04:00+08 | 6.886077 | 0.073783 | 0.038586  
18. node-3 | system.io.util | 30 | 4 | 25 | 2022-03-20 16:00:00+08 | 6.857143 | 0.350000 | 0.000000  
19. node-3 | system.io.w_s | 30 | 2 | 25 | 2022-03-20 16:01:00+08 | 6.441860 | 12.900000 | 0.000000  
20. emailservice-2 | mesh.destination.checkoutservice.emailservice.... | 30 | 2 | 3 | 2022-03-20 16:02:00+08 | 4.506941 | 0.007114 | 0.003150

Notes:
- The full per-group candidate table is stored as metric_fault_candidates in the kernel for follow-up analysis.
- Severity is derived from the maximum observed deviation ratio per group within the incident window; values above 0 indicate groups with measurable deviation beyond percentiles.

The original code execution output of IPython Kernel is also provided below for reference:

(   groups_with_any_anomaly  groups_with_positive_max_deviation
0                     4818                                1770,                  cmdb_id                                           kpi_name  count_in_window  anomaly_count_high  anomaly_count_low          earliest_high_ts           earliest_low_ts  max_dev_ratio_high  max_dev_ratio_low        p95        p5       earliest_anomaly_ts   severity
3085              node-1                                 system.io.avg_q_sz               30                  13                 17 2022-03-20 16:00:00+08:00 2022-03-20 16:03:00+08:00           25.000000       0.000000e+00   0.010000  0.000000 2022-03-20 16:00:00+08:00  25.000000
3532    paymentservice-1  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:20:00+08:00 2022-03-20 16:00:00+08:00           20.631559       5.894073e-03   0.025791  0.018894 2022-03-20 16:00:00+08:00  20.631559
3602    paymentservice-2  container.node-5.container_network_receive_MB....               30                   3                  2 2022-03-20 16:00:00+08:00 2022-03-20 16:15:00+08:00           20.322449       2.314761e-02   0.025848  0.019963 2022-03-20 16:00:00+08:00  20.322449
3466    paymentservice-0  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:13:00+08:00 2022-03-20 16:14:00+08:00           20.250528       3.857233e-02   0.025972  0.018995 2022-03-20 16:13:00+08:00  20.250528
3672   paymentservice2-0  container.node-6.container_network_receive_MB....               30                   2                  2 2022-03-20 16:23:00+08:00 2022-03-20 16:06:00+08:00           18.601700       1.830564e-01   0.028222  0.017363 2022-03-20 16:06:00+08:00  18.601700
2412     emailservice2-0  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:02:00+08:00 2022-03-20 16:00:00+08:00           18.320397       3.073875e-02   0.028764  0.019818 2022-03-20 16:00:00+08:00  18.320397
1585   checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...               30                   2                 21 2022-03-20 16:07:00+08:00 2022-03-20 16:00:00+08:00           15.265625       0.000000e+00   0.001600  0.000000 2022-03-20 16:00:00+08:00  15.265625
4618   shippingservice-1  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:10:00+08:00 2022-03-20 16:02:00+08:00           13.047716       1.425496e-02   0.046568  0.034712 2022-03-20 16:02:00+08:00  13.047716
4778  shippingservice2-0  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:20:00+08:00 2022-03-20 16:02:00+08:00           12.079747       8.362627e-02   0.042903  0.026187 2022-03-20 16:02:00+08:00  12.079747
1579   checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...               30                   2                  2 2022-03-20 16:23:00+08:00 2022-03-20 16:10:00+08:00           10.947642       6.040268e-02   0.005539  0.003911 2022-03-20 16:10:00+08:00  10.947642
4538   shippingservice-0  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:03:00+08:00 2022-03-20 16:19:00+08:00           10.511052       6.362220e-02   0.049265  0.036137 2022-03-20 16:03:00+08:00  10.511052
2215      emailservice-0  mesh.destination.checkoutservice.emailservice....               30                   2                  2 2022-03-20 16:06:00+08:00 2022-03-20 16:11:00+08:00           10.376623       4.530342e-13   0.005775  0.003675 2022-03-20 16:06:00+08:00  10.376623
1768  checkoutservice2-0  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:05:00+08:00 2022-03-20 16:03:00+08:00           10.063825       1.184443e-01   0.052091  0.035035 2022-03-20 16:03:00+08:00  10.063825
1532   checkoutservice-1  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:08:00+08:00 2022-03-20 16:11:00+08:00            8.413486       5.165669e-02   0.062166  0.042268 2022-03-20 16:08:00+08:00   8.413486
545          adservice-2  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:25:00+08:00 2022-03-20 16:20:00+08:00            7.441061       1.420501e-02   0.074807  0.057934 2022-03-20 16:20:00+08:00   7.441061
3202              node-3                                 system.io.avg_q_sz               30                   2                 28 2022-03-20 16:24:00+08:00 2022-03-20 16:00:00+08:00            7.363636       0.000000e+00   0.027500  0.000000 2022-03-20 16:00:00+08:00   7.363636
960         adservice2-0  container.node-6.container_network_receive_MB....               30                   2                  2 2022-03-20 16:21:00+08:00 2022-03-20 16:04:00+08:00            6.886077       1.152337e-01   0.073783  0.038586 2022-03-20 16:04:00+08:00   6.886077
3208              node-3                                     system.io.util               30                   4                 25 2022-03-20 16:00:00+08:00 2022-03-20 16:01:00+08:00            6.857143       0.000000e+00   0.350000  0.000000 2022-03-20 16:00:00+08:00   6.857143
3210              node-3                                      system.io.w_s               30                   2                 25 2022-03-20 16:24:00+08:00 2022-03-20 16:01:00+08:00            6.441860       0.000000e+00  12.900000  0.000000 2022-03-20 16:01:00+08:00   6.441860
2363      emailservice-2  mesh.destination.checkoutservice.emailservice....               30                   2                  3 2022-03-20 16:08:00+08:00 2022-03-20 16:02:00+08:00            4.506941       1.666667e-01   0.007114  0.003150 2022-03-20 16:02:00+08:00   4.506941)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse existing kernel variables: trace_df, trace_groups
tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df has timezone-aware datetime column in UTC+8
if '_ts_dt' not in trace_df.columns:
    trace_df['_ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window (inclusive start, exclusive end)
start_dt = tz.localize(datetime(2022,3,20,16,0,0))
end_dt   = tz.localize(datetime(2022,3,20,16,30,0))

# Filter trace rows to the window
trace_window = trace_df[(trace_df['_ts_dt'] >= start_dt) & (trace_df['_ts_dt'] < end_dt)].copy()

# Merge global thresholds (p95, p5) from trace_groups into trace_window
trace_window = trace_window.merge(
    trace_groups[['cmdb_id','trace_name','p95','p5']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# Mark high/low anomalies and compute per-row deviation ratios safely
trace_window['is_high'] = trace_window['value'] >= trace_window['p95']
trace_window['is_low']  = trace_window['value'] <= trace_window['p5']

trace_window['dev_high'] = 0.0
mask_high = trace_window['is_high'] & (trace_window['p95'] > 0)
trace_window.loc[mask_high, 'dev_high'] = (trace_window.loc[mask_high, 'value'] - trace_window.loc[mask_high, 'p95']) / trace_window.loc[mask_high, 'p95']

trace_window['dev_low'] = 0.0
mask_low = trace_window['is_low'] & (trace_window['p5'] > 0)
trace_window.loc[mask_low, 'dev_low'] = (trace_window.loc[mask_low, 'p5'] - trace_window.loc[mask_low, 'value']) / trace_window.loc[mask_low, 'p5']

# Helper to get earliest ts among rows where condition true inside group
def earliest_ts(series):
    if series.dropna().empty:
        return pd.NaT
    return series.min()

# Group by (cmdb_id, trace_name) and compute aggregates
agg = trace_window.groupby(['cmdb_id','trace_name']).agg(
    count_in_window = ('value','count'),
    anomaly_count_high = ('is_high','sum'),
    anomaly_count_low  = ('is_low','sum'),
    earliest_high_ts = ('_ts_dt', lambda s: earliest_ts(s[trace_window.loc[s.index,'is_high']])),
    earliest_low_ts  = ('_ts_dt', lambda s: earliest_ts(s[trace_window.loc[s.index,'is_low']])),
    max_dev_ratio_high = ('dev_high','max'),
    max_dev_ratio_low  = ('dev_low','max')
).reset_index()

# Replace NaT with None for compact display if needed, keep datetime objects for computing earliest_anomaly_ts
agg['earliest_high_ts'] = agg['earliest_high_ts'].where(agg['earliest_high_ts'].notna(), None)
agg['earliest_low_ts']  = agg['earliest_low_ts'].where(agg['earliest_low_ts'].notna(), None)

# Fill NaNs for dev ratios
agg['max_dev_ratio_high'] = agg['max_dev_ratio_high'].fillna(0).astype(float)
agg['max_dev_ratio_low']  = agg['max_dev_ratio_low'].fillna(0).astype(float)

# Store full per-group trace anomaly summary for follow-up
trace_anomaly_summary = agg.copy()

# 2) Aggregate counts
any_anom_mask = (trace_anomaly_summary['anomaly_count_high'] > 0) | (trace_anomaly_summary['anomaly_count_low'] > 0)
positive_dev_mask = (trace_anomaly_summary['max_dev_ratio_high'] > 0) | (trace_anomaly_summary['max_dev_ratio_low'] > 0)

total_trace_groups_with_any_anomaly = int(any_anom_mask.sum())
trace_groups_with_positive_dev = int((any_anom_mask & positive_dev_mask).sum())

counts_df = pd.DataFrame([{
    'trace_groups_with_any_anomaly': total_trace_groups_with_any_anomaly,
    'trace_groups_with_positive_max_deviation': trace_groups_with_positive_dev
}])

# 3) Top 20 by severity
# Join global p95/p5 from trace_groups for display
tg_small = trace_groups[['cmdb_id','trace_name','p95','p5']]
candidates = trace_anomaly_summary.merge(tg_small, on=['cmdb_id','trace_name'], how='left')

# Compute severity and earliest_anomaly_ts (use actual pd.Timestamp where available)
def to_ts(x):
    return pd.NaT if x is None else pd.to_datetime(x)

candidates['earliest_high_ts_ts'] = candidates['earliest_high_ts'].apply(to_ts)
candidates['earliest_low_ts_ts']  = candidates['earliest_low_ts'].apply(to_ts)

candidates['earliest_anomaly_ts'] = candidates[['earliest_high_ts_ts','earliest_low_ts_ts']].min(axis=1)
candidates['severity'] = candidates[['max_dev_ratio_high','max_dev_ratio_low']].max(axis=1)

cols_out = [
    'cmdb_id','trace_name','count_in_window',
    'anomaly_count_high','anomaly_count_low',
    'earliest_high_ts','earliest_low_ts',
    'p95','p5','earliest_anomaly_ts','severity'
]

trace_anomaly_summary = candidates.copy()  # store full
trace_anomaly_summary_top20 = trace_anomaly_summary.sort_values(
    by=['severity','earliest_anomaly_ts'],
    ascending=[False, True]
).head(20)[cols_out]

# Compact output: counts and top20
counts_df, trace_anomaly_summary_top20, trace_anomaly_summary

```
Out[6]:
```
Summary of trace-based anomaly detection for the incident window (2022-03-20 16:00–16:30 Asia/Shanghai):

1) Aggregate counts
- Total trace (cmdb_id, trace_name) groups with any anomaly in-window: 1,232
- Of those, groups with a positive max deviation (severity > 0): 840

2) Top candidates (severity = max dev ratio), highlights from the top 20
(Displayed columns computed: cmdb_id | trace_name | count_in_window | high_cnt | low_cnt | earliest_high_ts | earliest_low_ts | p95 | p5 | earliest_anomaly_ts | severity)

- cartservice-0 | trace.from_frontend-1.duration_mean — 30 pts | high=3 | low=21 | earliest_high=16:08 | earliest_low=16:00 | p95=0.000037 | p5=0.000000 | earliest_anomaly=16:00 | severity=26.027027
- frontend-1 | trace.to_cartservice-0.duration_mean — 30 | high=3 | low=21 | earliest_high=16:08 | earliest_low=16:00 | p95=0.000037 | p5=0.000000 | earliest_anomaly=16:00 | severity=26.027027
- cartservice-1 | trace.from_frontend-1.duration_mean — 30 | high=2 | low=21 | earliest_high=16:15 | earliest_low=16:00 | p95=0.000096 | p5=0.000000 | earliest_anomaly=16:00 | severity=19.230005
- frontend-1 | trace.to_cartservice-1.duration_mean — 30 | high=2 | low=21 | earliest_high=16:15 | earliest_low=16:00 | earliest_anomaly=16:00 | severity=19.230005
- cartservice-2 | trace.self.duration_mean — 30 | high=2 | low=26 | earliest_high=16:07 | earliest_low=16:00 | severity≈18.8856
- (other top entries include combinations of frontend ⇄ cartservice traces, checkout/payment traces, productcatalog/recommendation traces and several shippingservice traces)
- Many top-severity traces have earliest_low_ts = 2022-03-20 16:00:00+08 (anomalies start at the beginning of the window).

3) Observations / interpretation
- A large fraction of trace groups show anomalies and a substantial number (840) show measurable deviation beyond global percentiles — indicating widespread trace-level deviations during the window.
- The highest severities appear in frontend ↔ cartservice related traces (both directions) and some service-to-service duration metrics, suggesting trace latency/response abnormalities concentrated around cart-related flows and several payment/checkout interactions.
- Earliest anomalies for many top candidates occur at or near 16:00+08, with some high-side anomalies appearing later in the window (e.g., ~16:07–16:15).

4) Next steps
- The full per-group trace anomaly table is stored as trace_anomaly_summary in the kernel for deeper inspection (e.g., inspect raw time series for the top candidates, correlate with metric-level anomalies, or group by service to find systemic impact).

The original code execution output of IPython Kernel is also provided below for reference:

(   trace_groups_with_any_anomaly  trace_groups_with_positive_max_deviation
0                           1232                                       840,                       cmdb_id                                       trace_name  count_in_window  anomaly_count_high  anomaly_count_low          earliest_high_ts           earliest_low_ts       p95        p5       earliest_anomaly_ts   severity
56              cartservice-0              trace.from_frontend-1.duration_mean               30                   3                 21 2022-03-20 16:08:00+08:00 2022-03-20 16:00:00+08:00  0.000037  0.000000 2022-03-20 16:00:00+08:00  26.027027
680                frontend-1             trace.to_cartservice-0.duration_mean               30                   3                 21 2022-03-20 16:08:00+08:00 2022-03-20 16:00:00+08:00  0.000037  0.000000 2022-03-20 16:00:00+08:00  26.027027
84              cartservice-1              trace.from_frontend-1.duration_mean               30                   2                 21 2022-03-20 16:15:00+08:00 2022-03-20 16:00:00+08:00  0.000096  0.000000 2022-03-20 16:00:00+08:00  19.230005
684                frontend-1             trace.to_cartservice-1.duration_mean               30                   2                 21 2022-03-20 16:15:00+08:00 2022-03-20 16:00:00+08:00  0.000096  0.000000 2022-03-20 16:00:00+08:00  19.230005
120             cartservice-2                         trace.self.duration_mean               30                   2                 26 2022-03-20 16:07:00+08:00 2022-03-20 16:00:00+08:00  0.000035  0.000000 2022-03-20 16:00:00+08:00  18.885551
116             cartservice-2              trace.from_frontend-2.duration_mean               30                   2                 15 2022-03-20 16:07:00+08:00 2022-03-20 16:01:00+08:00  0.000161  0.000000 2022-03-20 16:01:00+08:00  12.582090
780                frontend-2             trace.to_cartservice-2.duration_mean               30                   2                 15 2022-03-20 16:07:00+08:00 2022-03-20 16:01:00+08:00  0.000161  0.000000 2022-03-20 16:01:00+08:00  12.582090
92              cartservice-1                         trace.self.duration_mean               30                   2                 26 2022-03-20 16:14:00+08:00 2022-03-20 16:00:00+08:00  0.000162  0.000000 2022-03-20 16:00:00+08:00   4.046325
385         checkoutservice-2           trace.to_paymentservice-2.duration_p95               30                   2                  2 2022-03-20 16:05:00+08:00 2022-03-20 16:11:00+08:00  0.000141  0.000124 2022-03-20 16:05:00+08:00   3.753108
913          paymentservice-2        trace.from_checkoutservice-2.duration_p95               30                   2                  2 2022-03-20 16:05:00+08:00 2022-03-20 16:11:00+08:00  0.000141  0.000124 2022-03-20 16:05:00+08:00   3.753108
1029  productcatalogservice-2  trace.from_recommendationservice-1.duration_p95               30                   2                  4 2022-03-20 16:11:00+08:00 2022-03-20 16:02:00+08:00  0.000038  0.000026 2022-03-20 16:02:00+08:00   3.556291
1101  recommendationservice-1    trace.to_productcatalogservice-2.duration_p95               30                   2                  4 2022-03-20 16:11:00+08:00 2022-03-20 16:02:00+08:00  0.000038  0.000026 2022-03-20 16:02:00+08:00   3.556291
60              cartservice-0              trace.from_frontend-2.duration_mean               30                   2                 18 2022-03-20 16:14:00+08:00 2022-03-20 16:00:00+08:00  0.000073  0.000000 2022-03-20 16:00:00+08:00   3.295486
772                frontend-2             trace.to_cartservice-0.duration_mean               30                   2                 18 2022-03-20 16:14:00+08:00 2022-03-20 16:00:00+08:00  0.000073  0.000000 2022-03-20 16:00:00+08:00   3.295486
405         checkoutservice-2          trace.to_shippingservice-1.duration_p95               30                   2                  5 2022-03-20 16:00:00+08:00 2022-03-20 16:03:00+08:00  0.000080  0.000057 2022-03-20 16:00:00+08:00   3.146949
1185        shippingservice-1        trace.from_checkoutservice-2.duration_p95               30                   2                  5 2022-03-20 16:00:00+08:00 2022-03-20 16:03:00+08:00  0.000080  0.000057 2022-03-20 16:00:00+08:00   3.146949
64              cartservice-0                         trace.self.duration_mean               30                   2                 27 2022-03-20 16:14:00+08:00 2022-03-20 16:00:00+08:00  0.000058  0.000000 2022-03-20 16:00:00+08:00   3.140291
269         checkoutservice-1          trace.to_currencyservice-2.duration_p95               30                   2                  2 2022-03-20 16:05:00+08:00 2022-03-20 16:09:00+08:00  0.000123  0.000100 2022-03-20 16:05:00+08:00   3.128927
501         currencyservice-2        trace.from_checkoutservice-1.duration_p95               30                   2                  2 2022-03-20 16:05:00+08:00 2022-03-20 16:09:00+08:00  0.000123  0.000100 2022-03-20 16:05:00+08:00   3.128927
437        checkoutservice2-0          trace.to_paymentservice2-0.duration_p95               27                   2                  3 2022-03-20 16:02:00+08:00 2022-03-20 16:09:00+08:00  0.000183  0.000146 2022-03-20 16:02:00+08:00   3.028462,                  cmdb_id                               trace_name  count_in_window  anomaly_count_high  anomaly_count_low          earliest_high_ts           earliest_low_ts  max_dev_ratio_high  max_dev_ratio_low        p95         p5       earliest_high_ts_ts        earliest_low_ts_ts       earliest_anomaly_ts  severity
0            adservice-0      trace.from_frontend-0.duration_mean               30                   2                 21 2022-03-20 16:01:00+08:00 2022-03-20 16:00:00+08:00            0.205674           0.000000   0.000014   0.000012 2022-03-20 16:01:00+08:00 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00  0.205674
1            adservice-0       trace.from_frontend-0.duration_p95               30                   3                  8 2022-03-20 16:01:00+08:00 2022-03-20 16:03:00+08:00            0.150000           0.000000   0.000020   0.000013 2022-03-20 16:01:00+08:00 2022-03-20 16:03:00+08:00 2022-03-20 16:01:00+08:00  0.150000
2            adservice-0         trace.from_frontend-0.error_rate               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00            0.000000           0.000000   0.000000   0.000000 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00  0.000000
3            adservice-0          trace.from_frontend-0.row_count               30                   2                  2 2022-03-20 16:11:00+08:00 2022-03-20 16:12:00+08:00            0.088929           0.024390  27.550000  18.450000 2022-03-20 16:11:00+08:00 2022-03-20 16:12:00+08:00 2022-03-20 16:11:00+08:00  0.088929
4            adservice-0      trace.from_frontend-1.duration_mean               30                   2                  2 2022-03-20 16:05:00+08:00 2022-03-20 16:03:00+08:00            0.433447           0.039301   0.000015   0.000011 2022-03-20 16:05:00+08:00 2022-03-20 16:03:00+08:00 2022-03-20 16:03:00+08:00  0.433447
...                  ...                                      ...              ...                 ...                ...                       ...                       ...                 ...                ...        ...        ...                       ...                       ...                       ...       ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count               27                   3                  2 2022-03-20 16:06:00+08:00 2022-03-20 16:05:00+08:00            0.200000           0.230769  10.000000   2.600000 2022-03-20 16:06:00+08:00 2022-03-20 16:05:00+08:00 2022-03-20 16:05:00+08:00  0.230769
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean               30                   2                  2 2022-03-20 16:05:00+08:00 2022-03-20 16:07:00+08:00            0.032864           0.024808   0.000075   0.000058 2022-03-20 16:05:00+08:00 2022-03-20 16:07:00+08:00 2022-03-20 16:05:00+08:00  0.032864
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95               30                   2                  4 2022-03-20 16:11:00+08:00 2022-03-20 16:02:00+08:00            0.029790           0.031746   0.000136   0.000063 2022-03-20 16:11:00+08:00 2022-03-20 16:02:00+08:00 2022-03-20 16:02:00+08:00  0.031746
1230  shippingservice2-0        trace.from_frontend2-0.error_rate               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00            0.000000           0.000000   0.000000   0.000000 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00  0.000000
1231  shippingservice2-0         trace.from_frontend2-0.row_count               30                   3                  2 2022-03-20 16:17:00+08:00 2022-03-20 16:16:00+08:00            0.000000           0.224806  20.000000   6.450000 2022-03-20 16:17:00+08:00 2022-03-20 16:16:00+08:00 2022-03-20 16:16:00+08:00  0.224806

[1232 rows x 15 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Filter metric_fault_candidates for cart & frontend components and prepare compact output
components = ['cartservice-0','cartservice-1','cartservice-2','cartservice2-0','frontend-0','frontend-1','frontend-2']

cart_frontend_metric_candidates = (
    metric_fault_candidates[metric_fault_candidates['cmdb_id'].isin(components)]
    .loc[:, [
        'cmdb_id','kpi_name','count_in_window',
        'anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_ts','max_dev_ratio_high','max_dev_ratio_low',
        'p95','p5','severity'
    ]]
    .sort_values(by='severity', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

cart_frontend_metric_candidates

```
Out[7]:
```
Summary (compact) for cart & frontend metric fault candidates (top ≤20 by severity):

- Scope: filtered metric_fault_candidates to cartservice-*/frontend-*; returned up to 20 highest-severity metric groups.

- Highest-severity findings:
  - cartservice2-0 | container.node-6.container_cpu_cfs_throttled_* (two related KPIs) — severity = 2.636. Window: 30 points; anomaly counts = high:2, low:28; earliest anomaly = 2022-03-20 16:00+08. This indicates notable CPU throttling signals for cartservice2-0 on node-6.
  - cartservice2-0 | container.node-6.container_network_receive_MB.* — severity ≈ 1.48; earliest anomaly ≈ 16:03.
  - frontend-0 / frontend-2 / frontend-1 | container.node-5.container_network_receive_MB.* — severities ~1.12, 0.41, 0.34 respectively; earliest anomalies in the 16:02–16:08 range. These are network receive I/O spikes at frontends.
  - Several cartservice-2 and cartservice-0 groups show memory_failures and cpu_throttled signals (severity ~1.10 and ~0.82), with earliest anomalies at 16:00–16:07.
  - Some mesh (service-to-service) metrics for cartservice2-0 show large p95/p5 magnitudes but moderate severity (0.41), meaning relative deviation exists though absolute scales vary.

- Pattern summary:
  - Metrics concentrate on container CPU throttling, container network receive rates, and container memory/failure counters — across cartservice pods and frontend pods.
  - Earliest anomalies for many top candidates start at or near 16:00+08 (the start of the incident window), with some specific spikes occurring later (~16:03–16:17).
  - Several groups show both high-side and low-side anomaly counts (both anomaly_count_high and anomaly_count_low > 0), reflecting threshold-equality or flat distributions in global percentiles for some KPIs.

- Suggested next steps:
  - Inspect raw time series for the top 3–5 candidates (start with cartservice2-0 CPU throttling and frontend/container network_receive metrics) to confirm sustained deviations vs single-point noise.
  - Correlate these metric candidates with trace-level top anomalies (cart/frontend traces) and error logs to identify impact on requests and root cause (CPU throttling vs network I/O).
  - If confirmed, investigate node-level resource contention on node-5/node-6 and pod resource/configuration (CPU limits, throttling, networking).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window  anomaly_count_high  anomaly_count_low       earliest_anomaly_ts  max_dev_ratio_high  max_dev_ratio_low          p95           p5  severity
0   cartservice2-0  container.node-6.container_cpu_cfs_throttled_p...               30                   2                 28 2022-03-20 16:00:00+08:00            2.636364           0.000000     0.366667     0.000000  2.636364
1   cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...               30                   2                 27 2022-03-20 16:00:00+08:00            2.636364           0.000000     0.035609     0.000000  2.636364
2   cartservice2-0  container.node-6.container_network_receive_MB....               30                   2                  2 2022-03-20 16:03:00+08:00            1.481457           0.040196     0.173384     0.046832  1.481457
3       frontend-0  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:08:00+08:00            1.118604           0.072323     0.463204     0.350176  1.118604
4    cartservice-2  container.node-5.container_memory_failures.hie...               30                   2                  2 2022-03-20 16:07:00+08:00            1.101901           0.027484   328.750000    59.125000  1.101901
5    cartservice-2  container.node-5.container_memory_failures.con...               30                   2                  2 2022-03-20 16:07:00+08:00            1.101901           0.027484   328.750000    59.125000  1.101901
6    cartservice-2  container.node-5.container_cpu_cfs_throttled_p...               30                   2                 28 2022-03-20 16:00:00+08:00            0.818182           0.000000     0.275000     0.000000  0.818182
7    cartservice-2  container.node-5.container_cpu_cfs_throttled_s...               30                   2                 28 2022-03-20 16:00:00+08:00            0.818182           0.000000     0.009352     0.000000  0.818182
8    cartservice-2  mesh.destination.frontend.cartservice.istio_re...               30                   2                  2 2022-03-20 16:02:00+08:00            0.675055           0.005293     0.111265     0.096586  0.675055
9    cartservice-1  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:03:00+08:00            0.569695           0.045961     0.232160     0.077203  0.569695
10   cartservice-2  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:06:00+08:00            0.532544           0.120502     0.236259     0.075081  0.532544
11   cartservice-0  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:04:00+08:00            0.522398           0.078872     0.237201     0.073331  0.522398
12  cartservice2-0  mesh.destination.checkoutservice2.cartservice2...               30                   7                  2 2022-03-20 16:04:00+08:00            0.142857           0.420290  4585.000000  2259.750000  0.420290
13  cartservice2-0  mesh.destination.checkoutservice2.cartservice2...               30                   2                  2 2022-03-20 16:05:00+08:00            0.142857           0.420290     0.007350     0.003622  0.420290
14  cartservice2-0  mesh.destination.checkoutservice2.cartservice2...               30                   5                  2 2022-03-20 16:04:00+08:00            0.142857           0.414414  9100.000000  4440.000000  0.414414
15      frontend-2  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:02:00+08:00            0.412179           0.128140     0.492016     0.360311  0.412179
16      frontend-1  container.node-5.container_network_receive_MB....               30                   2                  2 2022-03-20 16:17:00+08:00            0.337202           0.125737     0.520508     0.327371  0.337202
17      frontend-0  mesh.source.frontend.checkoutservice.istio_req...               30                   2                  2 2022-03-20 16:04:00+08:00            0.315041           0.060271     0.272425     0.140200  0.315041
18      frontend-0  mesh.source.frontend.shippingservice.istio_req...               30                   2                  2 2022-03-20 16:00:00+08:00            0.058738           0.299720    34.475000    17.850000  0.299720
19      frontend-0  mesh.source.frontend.shippingservice.istio_req...               30                   2                  2 2022-03-20 16:00:00+08:00            0.058738           0.299720    34.475000    17.850000  0.299720```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime, timedelta

# Reuse variables from kernel: metric_df, metric_groups
tz = pytz.timezone('Asia/Shanghai')

# Target cmdb_ids and substring
cmdb_list = ['cartservice2-0','cartservice-2','cartservice-1','cartservice-0']
substr = 'container_cpu_cfs_throttled'

# Ensure timestamp dt column exists
if '_ts_dt' not in metric_df.columns:
    metric_df['_ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window
start_dt = tz.localize(datetime(2022,3,20,16,0,0))
end_dt   = tz.localize(datetime(2022,3,20,16,30,0))

# 1) Find matching groups from metric_groups
mg_sub = metric_groups[
    metric_groups['cmdb_id'].isin(cmdb_list) &
    metric_groups['kpi_name'].str.contains(substr, regex=False)
].loc[:, ['cmdb_id','kpi_name','p95','p5']].reset_index(drop=True)

# If no matches, return empty results
if mg_sub.empty:
    cart_cpu_throttle_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','p95','p5','count_in_window',
        'earliest_high_ts','high_run_len_min','earliest_low_ts','low_run_len_min'
    ])
    cart_cpu_throttle_samples = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value'])
    cart_cpu_throttle_series = cart_cpu_throttle_samples.copy()
else:
    # 2) Extract full metric points within window for these groups and merge thresholds
    # Merge thresholds into metric_df then filter
    mg_small = mg_sub.copy()
    merged = metric_df.merge(mg_small, on=['cmdb_id','kpi_name'], how='inner')
    window_rows = merged[(merged['_ts_dt'] >= start_dt) & (merged['_ts_dt'] < end_dt)].copy()
    # Keep full series stored for follow-up
    cart_cpu_throttle_series = window_rows.sort_values(['cmdb_id','kpi_name','_ts_dt']).reset_index(drop=True)

    # Helper to compute earliest run start and length (minute granularity)
    def compute_first_run(df_group, thresh_col, op='ge'):
        # df_group expected sorted by _ts_dt ascending
        if df_group.empty:
            return None, 0
        # minute index (int)
        minutes = (df_group['timestamp'] // 60).astype(int)
        vals = df_group['value'].values
        mins = minutes.values
        # determine mask for condition
        if op == 'ge':
            mask = df_group['value'].values >= df_group[thresh_col].values
        else:
            mask = df_group['value'].values <= df_group[thresh_col].values
        if not mask.any():
            return None, 0
        # find earliest minute index among masked rows
        masked_minutes = mins[mask]
        earliest_min = int(masked_minutes.min())
        # Build set of masked minute values for quick membership
        masked_min_set = set(np.unique(masked_minutes))
        # Count consecutive minutes starting from earliest_min
        run_len = 0
        cur = earliest_min
        while cur in masked_min_set:
            run_len += 1
            cur += 1  # next minute
        # convert earliest_min back to timestamp (seconds) and to tz-aware dt
        earliest_ts = pd.to_datetime(earliest_min * 60, unit='s', utc=True).tz_convert(tz)
        return earliest_ts, run_len

    # Group and compute summaries
    summaries = []
    samples_list = []
    for (cmdb, kpi), grp in cart_cpu_throttle_series.groupby(['cmdb_id','kpi_name'], sort=False):
        grp_sorted = grp.sort_values('_ts_dt').reset_index(drop=True)
        p95 = float(grp_sorted['p95'].iloc[0])
        p5  = float(grp_sorted['p5'].iloc[0])
        count_in_window = int(grp_sorted.shape[0])

        # compute earliest high run and length
        earliest_high_ts, high_run_len = compute_first_run(grp_sorted, 'p95', op='ge')
        earliest_low_ts, low_run_len = compute_first_run(grp_sorted, 'p5', op='le')

        # prepare up to 30 sample rows for display: timestamp and value sorted
        samples = grp_sorted.loc[:, ['timestamp','_ts_dt','value']].copy().sort_values('_ts_dt').head(30)
        # convert timestamp to int seconds (already) and keep
        samples = samples.rename(columns={'_ts_dt':'_ts_dt_tz'})
        samples_list.append(
            pd.concat([
                pd.DataFrame({'cmdb_id':[cmdb]*len(samples),'kpi_name':[kpi]*len(samples)}).reset_index(drop=True),
                samples.reset_index(drop=True)
            ], axis=1)
        )

        summaries.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'p95': p95,
            'p5': p5,
            'count_in_window': count_in_window,
            'earliest_high_ts': earliest_high_ts,
            'high_run_len_min': high_run_len,
            'earliest_low_ts': earliest_low_ts,
            'low_run_len_min': low_run_len
        })

    cart_cpu_throttle_summary = pd.DataFrame(summaries).sort_values(['cmdb_id','kpi_name']).reset_index(drop=True)

    # Concatenate sample rows (limited up to 30 per group)
    if samples_list:
        cart_cpu_throttle_samples = pd.concat(samples_list, ignore_index=True)
        # tidy display: convert _ts_dt_tz to timezone string
        cart_cpu_throttle_samples['_ts_dt_tz'] = cart_cpu_throttle_samples['_ts_dt_tz'].dt.tz_convert(tz)
        cart_cpu_throttle_samples = cart_cpu_throttle_samples.rename(columns={'_ts_dt_tz':'_ts_dt'})
        # Keep columns as requested: timestamp, value (and group identifiers)
        cart_cpu_throttle_samples = cart_cpu_throttle_samples[['cmdb_id','kpi_name','timestamp','_ts_dt','value']]
    else:
        cart_cpu_throttle_samples = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','_ts_dt','value'])

# Store full per-group results in kernel variable (cart_cpu_throttle_series already set)
cart_cpu_throttle_summary, cart_cpu_throttle_samples, cart_cpu_throttle_series

```
Out[8]:
```
Summary (compact) of container_cpu_cfs_throttled metrics for cartservice pods during 2022-03-20 16:00–16:30 Asia/Shanghai:

Matched groups (8 groups total — two KPI variants per pod):

- cartservice-0
  - container.node-5.container_cpu_cfs_throttled_periods: p95 = 0.0, p5 = 0.0, 30 points in window. 
    - earliest value >= p95: 2022-03-20 16:00:00+08, first consecutive high run = 30 minutes.
    - earliest value <= p5: 2022-03-20 16:00:00+08, first consecutive low run = 30 minutes.
  - container.node-5.container_cpu_cfs_throttled_seconds: p95 = 0.0, p5 = 0.0, 30 points.
    - earliest_high = 16:00, high run = 30 min; earliest_low = 16:00, low run = 30 min.

- cartservice-1
  - container.node-5.container_cpu_cfs_throttled_periods: p95 = 0.0, p5 = 0.0, 30 points.
    - earliest_high = 16:00, high run = 30 min; earliest_low = 16:00, low run = 30 min.
  - container.node-5.container_cpu_cfs_throttled_seconds: p95 = 0.0, p5 = 0.0, 30 points.
    - earliest_high = 16:00, high run = 30 min; earliest_low = 16:00, low run = 30 min.

- cartservice-2
  - container.node-5.container_cpu_cfs_throttled_periods: p95 = 0.275, p5 = 0.0, 30 points.
    - earliest_high = 2022-03-20 16:23:00+08, first high run length = 2 minutes.
    - earliest_low = 2022-03-20 16:00:00+08, first low run length = 23 minutes.
  - container.node-5.container_cpu_cfs_throttled_seconds: p95 = 0.009352, p5 = 0.0, 30 points.
    - earliest_high = 16:23:00+08, high run = 2 min; earliest_low = 16:00:00+08, low run = 23 min.

- cartservice2-0
  - container.node-6.container_cpu_cfs_throttled_periods: p95 = 0.366667, p5 = 0.0, 30 points.
    - earliest_high = 2022-03-20 16:07:00+08, high run = 2 minutes.
    - earliest_low = 2022-03-20 16:00:00+08, low run = 7 minutes.
  - container.node-6.container_cpu_cfs_throttled_seconds: p95 = 0.035609, p5 = 0.0, 30 points.
    - earliest_high = 16:07:00+08, high run = 2 min; earliest_low = 16:00:00+08, low run = 2 min.

Additional notes and interpretation:
- Many groups for cartservice-0 and cartservice-1 have p95 == p5 == 0.0 and all 30 window points equal 0.0. That produces both "high" and "low" flags (value >= p95 and value <= p5) and yields runs of length 30 for both sides — this is an artifact of zero thresholds (no historical spread).
- cartservice-2 and cartservice2-0 show non-zero p95 values: they had short high spikes near the mid/late window (first detected ~16:23 for cartservice-2, ~16:07 for cartservice2-0) lasting ~2 minutes, while prolonged low sequences (zeros) occupy earlier minutes.
- The full per-minute series for these groups within the window was recorded (240 rows total across groups) and is stored in the kernel variable cart_cpu_throttle_series; a per-group sample (up to 30 rows each) is stored in cart_cpu_throttle_samples; summary table is cart_cpu_throttle_summary.

Suggested follow-up:
- Inspect the actual non-zero spike values at the earliest_high timestamps for cartservice-2 and cartservice2-0 to confirm significance.
- Treat p95==p5==0 groups carefully: consider alternative thresholding (e.g., historical smoothing or absolute thresholds) to avoid false dual-flagging when metrics are constant at zero.

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                           kpi_name       p95   p5  count_in_window          earliest_high_ts  high_run_len_min           earliest_low_ts  low_run_len_min
0   cartservice-0  container.node-5.container_cpu_cfs_throttled_p...  0.000000  0.0               30 2022-03-20 16:00:00+08:00                30 2022-03-20 16:00:00+08:00               30
1   cartservice-0  container.node-5.container_cpu_cfs_throttled_s...  0.000000  0.0               30 2022-03-20 16:00:00+08:00                30 2022-03-20 16:00:00+08:00               30
2   cartservice-1  container.node-5.container_cpu_cfs_throttled_p...  0.000000  0.0               30 2022-03-20 16:00:00+08:00                30 2022-03-20 16:00:00+08:00               30
3   cartservice-1  container.node-5.container_cpu_cfs_throttled_s...  0.000000  0.0               30 2022-03-20 16:00:00+08:00                30 2022-03-20 16:00:00+08:00               30
4   cartservice-2  container.node-5.container_cpu_cfs_throttled_p...  0.275000  0.0               30 2022-03-20 16:23:00+08:00                 2 2022-03-20 16:00:00+08:00               23
5   cartservice-2  container.node-5.container_cpu_cfs_throttled_s...  0.009352  0.0               30 2022-03-20 16:23:00+08:00                 2 2022-03-20 16:00:00+08:00               23
6  cartservice2-0  container.node-6.container_cpu_cfs_throttled_p...  0.366667  0.0               30 2022-03-20 16:07:00+08:00                 2 2022-03-20 16:00:00+08:00                7
7  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...  0.035609  0.0               30 2022-03-20 16:07:00+08:00                 2 2022-03-20 16:00:00+08:00                2,             cmdb_id                                           kpi_name   timestamp                    _ts_dt  value
0     cartservice-0  container.node-5.container_cpu_cfs_throttled_p...  1647763200 2022-03-20 16:00:00+08:00    0.0
1     cartservice-0  container.node-5.container_cpu_cfs_throttled_p...  1647763260 2022-03-20 16:01:00+08:00    0.0
2     cartservice-0  container.node-5.container_cpu_cfs_throttled_p...  1647763320 2022-03-20 16:02:00+08:00    0.0
3     cartservice-0  container.node-5.container_cpu_cfs_throttled_p...  1647763380 2022-03-20 16:03:00+08:00    0.0
4     cartservice-0  container.node-5.container_cpu_cfs_throttled_p...  1647763440 2022-03-20 16:04:00+08:00    0.0
..              ...                                                ...         ...                       ...    ...
235  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...  1647764700 2022-03-20 16:25:00+08:00    0.0
236  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...  1647764760 2022-03-20 16:26:00+08:00    0.0
237  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...  1647764820 2022-03-20 16:27:00+08:00    0.0
238  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...  1647764880 2022-03-20 16:28:00+08:00    0.0
239  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...  1647764940 2022-03-20 16:29:00+08:00    0.0

[240 rows x 5 columns],       timestamp         cmdb_id                                           kpi_name  value                    _ts_dt       p95   p5
0    1647763200   cartservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.0 2022-03-20 16:00:00+08:00  0.000000  0.0
1    1647763260   cartservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.0 2022-03-20 16:01:00+08:00  0.000000  0.0
2    1647763320   cartservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.0 2022-03-20 16:02:00+08:00  0.000000  0.0
3    1647763380   cartservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.0 2022-03-20 16:03:00+08:00  0.000000  0.0
4    1647763440   cartservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.0 2022-03-20 16:04:00+08:00  0.000000  0.0
..          ...             ...                                                ...    ...                       ...       ...  ...
235  1647764700  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...    0.0 2022-03-20 16:25:00+08:00  0.035609  0.0
236  1647764760  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...    0.0 2022-03-20 16:26:00+08:00  0.035609  0.0
237  1647764820  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...    0.0 2022-03-20 16:27:00+08:00  0.035609  0.0
238  1647764880  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...    0.0 2022-03-20 16:28:00+08:00  0.035609  0.0
239  1647764940  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...    0.0 2022-03-20 16:29:00+08:00  0.035609  0.0

[240 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing kernel variables: log_df, trace_anomaly_summary
tz = pytz.timezone('Asia/Shanghai')

# Define incident window (inclusive start, exclusive end)
start_dt = tz.localize(datetime(2022,3,20,16,0,0))
end_dt   = tz.localize(datetime(2022,3,20,16,30,0))

# Ensure log_df has timezone-aware datetime column
if '_ts_dt' not in log_df.columns:
    log_df['_ts_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Filter logs for cartservice2-0 in the window
cartservice2_logs_all = log_df[
    (log_df['cmdb_id'] == 'cartservice2-0') &
    (log_df['_ts_dt'] >= start_dt) &
    (log_df['_ts_dt'] < end_dt)
].copy()

# Prepare up to 30 rows with requested columns sorted by timestamp
cartservice2_logs = cartservice2_logs_all.sort_values('_ts_dt').loc[:, ['timestamp','log_name','value']].head(30).reset_index(drop=True)

# Aggregates for log.error_count
log_err = cartservice2_logs_all[cartservice2_logs_all['log_name'] == 'log.error_count']
if not log_err.empty:
    sum_error_count = float(log_err['value'].sum())
    # earliest timestamp where log.error_count > 0
    err_pos = log_err[log_err['value'] > 0]
    earliest_error_ts = err_pos['_ts_dt'].min() if not err_pos.empty else None
else:
    sum_error_count = 0.0
    earliest_error_ts = None

cartservice2_log_aggs = pd.DataFrame([{
    'cmdb_id': 'cartservice2-0',
    'sum_log_error_count': sum_error_count,
    'earliest_log_error_ts': earliest_error_ts
}])

# 2) From trace_anomaly_summary, select rows for cartservice2-0 sorted by severity desc (limit 20)
# trace_anomaly_summary was stored earlier and contains p95/p5,severity,earliest_anomaly_ts, etc.
cartservice2_traces = trace_anomaly_summary[trace_anomaly_summary['cmdb_id'] == 'cartservice2-0'].copy()

# Select and order requested columns, sort by severity desc and limit 20
cols = [
    'trace_name','count_in_window','anomaly_count_high','anomaly_count_low',
    'earliest_high_ts','earliest_low_ts','p95','p5','earliest_anomaly_ts','severity'
]
# Some rows might not have all columns due to earlier merges; ensure presence
available_cols = [c for c in cols if c in cartservice2_traces.columns]
cartservice2_traces = cartservice2_traces.loc[:, available_cols] \
    .sort_values(by='severity', ascending=False) \
    .head(20) \
    .reset_index(drop=True)

# Keep outputs compact: return the two tables and the aggregate row
cartservice2_logs, cartservice2_log_aggs, cartservice2_traces

```
Out[9]:
```
Summary (compact) for cartservice2-0 within 2022-03-20 16:00–16:30 Asia/Shanghai:

Logs
- No log rows found for cmdb_id = cartservice2-0 in log.csv within the incident window.
- Sum of log.error_count = 0.0 and there is no earliest log.error_count > 0 (none observed).
- Stored variables: cartservice2_logs (empty result) and cartservice2_log_aggs (sum = 0.0, earliest = None).

Traces
- Returned up to 20 trace groups for cartservice2-0 (12 rows were returned, sorted by severity).
- Top trace candidates (highlights):
  - trace.self.duration_mean — 30 pts, high=2 / low=20, earliest_high=16:03, earliest_low=16:00, p95=0.000048, p5=0.0, severity=0.323529
  - trace.from_checkoutservice2-0.row_count — 27 pts, high=3 / low=2, earliest_high=16:06, earliest_low=16:05, p95=10.0, p5=2.6, severity=0.230769
  - trace.from_checkoutservice2-0.duration_mean — 27 pts, high=2 / low=17, earliest_high=16:05, earliest_low=16:00, p95=0.000425, p5=0.0, severity=0.176471
  - trace.from_frontend2-0.duration_mean and trace.from_frontend2-0.row_count — show elevated duration/row_count anomalies (severity ~0.11 and ~0.059), with earliest anomalies starting at or near 16:00–16:06.
  - Several traces (e.g., error_rate, duration_p95) show many high/low counts but zero severity (p95==p5 or zero thresholds), i.e., flagged but no positive deviation.
- Earliest anomalies for many top traces occur at or near 16:00+08; some high-side spikes appear a few minutes later (e.g., 16:03–16:06).
- Stored variable: cartservice2_traces contains the full returned trace-candidate table for follow-up.

Interpretation / next step suggestions
- No log error evidence in logs for cartservice2-0 during the window.
- Trace evidence indicates latency/row-count anomalies on cartservice2-0 (notably trace.self.duration_mean and row_count from checkoutservice flows) starting at or soon after 16:00.
- Recommend inspecting raw trace time series and correlating these top trace candidates with the metric-level CPU/network anomalies previously found for cartservice2-0 (and related services) to determine impact and root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp, log_name, value]
Index: [],           cmdb_id  sum_log_error_count earliest_log_error_ts
0  cartservice2-0                  0.0                  None,                                      trace_name  count_in_window  anomaly_count_high  anomaly_count_low          earliest_high_ts           earliest_low_ts        p95         p5       earliest_anomaly_ts  severity
0                      trace.self.duration_mean               30                   2                 20 2022-03-20 16:03:00+08:00 2022-03-20 16:00:00+08:00   0.000048   0.000000 2022-03-20 16:00:00+08:00  0.323529
1       trace.from_checkoutservice2-0.row_count               27                   3                  2 2022-03-20 16:06:00+08:00 2022-03-20 16:05:00+08:00  10.000000   2.600000 2022-03-20 16:05:00+08:00  0.230769
2   trace.from_checkoutservice2-0.duration_mean               27                   2                 17 2022-03-20 16:05:00+08:00 2022-03-20 16:00:00+08:00   0.000425   0.000000 2022-03-20 16:00:00+08:00  0.176471
3          trace.from_frontend2-0.duration_mean               30                   2                  2 2022-03-20 16:00:00+08:00 2022-03-20 16:06:00+08:00   0.000387   0.000126 2022-03-20 16:00:00+08:00  0.108098
4              trace.from_frontend2-0.row_count               30                   3                  2 2022-03-20 16:21:00+08:00 2022-03-20 16:06:00+08:00  73.000000  48.900000 2022-03-20 16:06:00+08:00  0.059305
5                          trace.self.row_count               30                   2                  2 2022-03-20 16:21:00+08:00 2022-03-20 16:04:00+08:00  88.400000  57.450000 2022-03-20 16:04:00+08:00  0.042646
6    trace.from_checkoutservice2-0.duration_p95               27                   2                 17 2022-03-20 16:06:00+08:00 2022-03-20 16:00:00+08:00   0.000985   0.000000 2022-03-20 16:00:00+08:00  0.015228
7      trace.from_checkoutservice2-0.error_rate               27                  27                 27 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00   0.000000   0.000000 2022-03-20 16:00:00+08:00  0.000000
8             trace.from_frontend2-0.error_rate               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00   0.000000   0.000000 2022-03-20 16:00:00+08:00  0.000000
9           trace.from_frontend2-0.duration_p95               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00   0.001000   0.001000 2022-03-20 16:00:00+08:00  0.000000
10                      trace.self.duration_p95               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00   0.000000   0.000000 2022-03-20 16:00:00+08:00  0.000000
11                        trace.self.error_rate               30                  30                 30 2022-03-20 16:00:00+08:00 2022-03-20 16:00:00+08:00   0.000000   0.000000 2022-03-20 16:00:00+08:00  0.000000)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
from datetime import datetime

# reuse kernel variables: metric_df, trace_df, metric_groups, trace_groups
tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware datetime columns exist
if '_ts_dt' not in metric_df.columns:
    metric_df['_ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if '_ts_dt' not in trace_df.columns:
    trace_df['_ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Metric rows for cartservice2-0, throttled KPIs, window 16:05 <= t < 16:10
win_start = tz.localize(datetime(2022,3,20,16,5,0))
win_end   = tz.localize(datetime(2022,3,20,16,10,0))

metric_mask = (
    (metric_df['cmdb_id'] == 'cartservice2-0') &
    (metric_df['kpi_name'].str.contains('container_cpu_cfs_throttled', regex=False)) &
    (metric_df['_ts_dt'] >= win_start) &
    (metric_df['_ts_dt'] < win_end)
)
cartservice2_metric_window = metric_df.loc[metric_mask, ['timestamp','kpi_name','value']].sort_values(['timestamp','kpi_name']).head(50).reset_index(drop=True)

# 2) Trace rows for cartservice2-0 for selected trace_names in same window
trace_names_want = [
    'trace.self.duration_mean',
    'trace.from_frontend2-0.duration_mean',
    'trace.from_checkoutservice2-0.duration_mean'
]
# filter to those present
available_trace_names = set(trace_df['trace_name'].unique())
trace_names = [n for n in trace_names_want if n in available_trace_names]

trace_mask = (
    (trace_df['cmdb_id'] == 'cartservice2-0') &
    (trace_df['trace_name'].isin(trace_names)) &
    (trace_df['_ts_dt'] >= win_start) &
    (trace_df['_ts_dt'] < win_end)
)
cartservice2_trace_window = trace_df.loc[trace_mask, ['timestamp','trace_name','value']].sort_values(['timestamp','trace_name']).head(50).reset_index(drop=True)

# 3) One-line summary:
# 3a) earliest timestamp within 16:00-16:30 where container_cpu_cfs_throttled_seconds >= global p95
full_start = tz.localize(datetime(2022,3,20,16,0,0))
full_end   = tz.localize(datetime(2022,3,20,16,30,0))

# find the exact kpi_name for throttled_seconds for cartservice2-0 from metric_groups
kpi_seconds_mask = metric_groups[
    (metric_groups['cmdb_id']=='cartservice2-0') &
    metric_groups['kpi_name'].str.contains('container_cpu_cfs_throttled_seconds', regex=False)
]
if not kpi_seconds_mask.empty:
    kpi_seconds_name = kpi_seconds_mask['kpi_name'].iloc[0]
    p95_seconds = float(kpi_seconds_mask['p95'].iloc[0])
    # filter metric_df for that kpi in full window
    mmask = (
        (metric_df['cmdb_id']=='cartservice2-0') &
        (metric_df['kpi_name']==kpi_seconds_name) &
        (metric_df['_ts_dt'] >= full_start) &
        (metric_df['_ts_dt'] < full_end)
    )
    seq = metric_df.loc[mmask, ['timestamp','_ts_dt','value']].sort_values('_ts_dt')
    cond = seq['value'] >= p95_seconds
    if cond.any():
        idx = cond.idxmax()  # first True index
        metric_earliest_ts = seq.loc[idx, '_ts_dt']
        metric_earliest_val = float(seq.loc[idx, 'value'])
    else:
        metric_earliest_ts = None
        metric_earliest_val = None
else:
    kpi_seconds_name = None
    p95_seconds = None
    metric_earliest_ts = None
    metric_earliest_val = None

# 3b) earliest timestamp within 16:00-16:30 where trace.self.duration_mean >= its global p95
trace_p95_mask = trace_groups[
    (trace_groups['cmdb_id']=='cartservice2-0') &
    (trace_groups['trace_name']=='trace.self.duration_mean')
]
if not trace_p95_mask.empty:
    trace_p95 = float(trace_p95_mask['p95'].iloc[0])
    tmask = (
        (trace_df['cmdb_id']=='cartservice2-0') &
        (trace_df['trace_name']=='trace.self.duration_mean') &
        (trace_df['_ts_dt'] >= full_start) &
        (trace_df['_ts_dt'] < full_end)
    )
    tseq = trace_df.loc[tmask, ['timestamp','_ts_dt','value']].sort_values('_ts_dt')
    tcond = tseq['value'] >= trace_p95
    if tcond.any():
        tidx = tcond.idxmax()
        trace_earliest_ts = tseq.loc[tidx,'_ts_dt']
        trace_earliest_val = float(tseq.loc[tidx,'value'])
    else:
        trace_earliest_ts = None
        trace_earliest_val = None
else:
    trace_p95 = None
    trace_earliest_ts = None
    trace_earliest_val = None

# Build compact one-line summary DataFrame
summary = pd.DataFrame([{
    'kpi_seconds_name': kpi_seconds_name,
    'kpi_seconds_p95': p95_seconds,
    'earliest_throttled_seconds_ge_p95_ts': metric_earliest_ts,
    'earliest_throttled_seconds_value': metric_earliest_val,
    'trace_name_checked': 'trace.self.duration_mean',
    'trace_p95': trace_p95,
    'earliest_trace_self_duration_mean_ge_p95_ts': trace_earliest_ts,
    'earliest_trace_self_duration_mean_value': trace_earliest_val
}])

# Store as requested for follow-up
cartservice2_correlation_check = {
    'metric_window_rows': cartservice2_metric_window,
    'trace_window_rows': cartservice2_trace_window,
    'summary': summary
}

# Display compact outputs: metric rows, trace rows, and the one-line summary
cartservice2_metric_window, cartservice2_trace_window, summary

```
Out[10]:
```
Compact summary of the correlation check for cartservice2-0 (window 2022-03-20 16:05–16:10 Asia/Shanghai):

1) Metric (container_cpu_cfs_throttled _periods and _seconds) — minutes in 16:05–16:09:
- 16:05 — periods=0.0, seconds=0.0
- 16:06 — periods=0.0, seconds=0.0
- 16:07 — periods=0.666667, seconds=0.064744  ← first non‑zero spike in this sub-window
- 16:08 — periods=1.333333, seconds=0.129488  ← larger spike
- 16:09 — periods=0.0, seconds=0.0

2) Traces (selected) in 16:05–16:09:
- trace.from_checkoutservice2-0.duration_mean: 16:05=0.0005, 16:06=0.0002, 16:08=0.0, ...
- trace.from_frontend2-0.duration_mean: 16:05=0.000311, 16:06=0.000125, 16:07=0.000203, 16:08=0.000118, ...
- trace.self.duration_mean: mostly 0.0 at 16:05–16:07, small nonzero at 16:08 (0.000016)

3) Earliest threshold crossings (within full 16:00–16:30 window):
- Earliest container_cpu_cfs_throttled_seconds >= global p95: 2022-03-20 16:07:00+08, value = 0.064744.
- Earliest trace.self.duration_mean >= global p95: 2022-03-20 16:03:00+08, value = 0.000063.

Interpretation
- A trace-level duration anomaly (trace.self.duration_mean) crossed its global p95 at ~16:03, several minutes before the CPU throttling spike observed at ~16:07.
- Within 16:05–16:10 the CPU throttling spike occurs at 16:07–16:08, concurrent with modest increases in some trace-duration metrics (e.g., trace.from_frontend2-0 at 16:07).
- This timing suggests trace latency increases began earlier (≈16:03) and the measurable CPU throttling on cartservice2-0 appears a few minutes later (≈16:07). That pattern is consistent with latency issues preceding (and possibly contributing to) the observed CPU throttling, but further causal analysis is required.

Stored results for follow-up in kernel:
- cartservice2_correlation_check['metric_window_rows'] (metric rows for 16:05–16:10),
- cartservice2_correlation_check['trace_window_rows'] (trace rows for 16:05–16:10),
- cartservice2_correlation_check['summary'] (one-line earliest-threshold summary).

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp                                           kpi_name     value
0  1647763500  container.node-6.container_cpu_cfs_throttled_p...  0.000000
1  1647763500  container.node-6.container_cpu_cfs_throttled_s...  0.000000
2  1647763560  container.node-6.container_cpu_cfs_throttled_p...  0.000000
3  1647763560  container.node-6.container_cpu_cfs_throttled_s...  0.000000
4  1647763620  container.node-6.container_cpu_cfs_throttled_p...  0.666667
5  1647763620  container.node-6.container_cpu_cfs_throttled_s...  0.064744
6  1647763680  container.node-6.container_cpu_cfs_throttled_p...  1.333333
7  1647763680  container.node-6.container_cpu_cfs_throttled_s...  0.129488
8  1647763740  container.node-6.container_cpu_cfs_throttled_p...  0.000000
9  1647763740  container.node-6.container_cpu_cfs_throttled_s...  0.000000,      timestamp                                   trace_name     value
0   1647763500  trace.from_checkoutservice2-0.duration_mean  0.000500
1   1647763500         trace.from_frontend2-0.duration_mean  0.000311
2   1647763500                     trace.self.duration_mean  0.000000
3   1647763560  trace.from_checkoutservice2-0.duration_mean  0.000200
4   1647763560         trace.from_frontend2-0.duration_mean  0.000125
5   1647763560                     trace.self.duration_mean  0.000000
6   1647763620         trace.from_frontend2-0.duration_mean  0.000203
7   1647763620                     trace.self.duration_mean  0.000000
8   1647763680  trace.from_checkoutservice2-0.duration_mean  0.000000
9   1647763680         trace.from_frontend2-0.duration_mean  0.000118
10  1647763680                     trace.self.duration_mean  0.000016
11  1647763740  trace.from_checkoutservice2-0.duration_mean  0.000000
12  1647763740         trace.from_frontend2-0.duration_mean  0.000290
13  1647763740                     trace.self.duration_mean  0.000000,                                     kpi_seconds_name  kpi_seconds_p95 earliest_throttled_seconds_ge_p95_ts  earliest_throttled_seconds_value        trace_name_checked  trace_p95 earliest_trace_self_duration_mean_ge_p95_ts  earliest_trace_self_duration_mean_value
0  container.node-6.container_cpu_cfs_throttled_s...         0.035609            2022-03-20 16:07:00+08:00                          0.064744  trace.self.duration_mean   0.000048                   2022-03-20 16:03:00+08:00                                 0.000063)```
```